In [1]:
#kmeans imports:

import os
import gc
import multiprocessing

import irisreader as ir
ir.config.verbosity_level=0
from irisreader.data.mg2k_centroids import get_mg2k_centroid_table, normalize
from irisreader import observation
from irisreader.coalignment import find_closest_raster

from IPython.display import HTML, display, clear_output
from collections import Counter

from sunpy.net import Fido
from sunpy.net import attrs as a
from tqdm import tqdm
from sunpy.time import parse_time
from sunpy.timeseries import TimeSeries
from astropy.time import Time

from matplotlib import animation
import matplotlib
from matplotlib.patches import Patch
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap

import sklearn
from sklearn.datasets import make_blobs
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestCentroid

from matplotlib.ticker import MaxNLocator
import matplotlib.gridspec as gridspec
from IPython.display import Image
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances_argmin_min

from scipy.spatial import distance_matrix
import scipy.io
import scipy

import numpy as np
np.random.seed(0)

import pandas as pd

import warnings
warnings.filterwarnings("ignore")

import utils_kmeans #kmeans utils
import som #som = self organising map class, + functions
import utils_data_prep # jonas data prep code utils
import utils_features as uf # jonas feature extraction code utils
import datetime
from datetime import datetime, timezone, timedelta
import pytz
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams['savefig.dpi'] = 600  # Set the desired pixels #standard is 300

# constants that are used throughout the script.
# better keep them together here, otherwise fcts are overpopulated with arguments
global_constants = {
    'fallback_colour': 'grey', #colour to plot if the cluster is not interesting / should not be seen.
    'global_triple_weight_factor': 3, #factor to do the triple weighing with
    'global_save_path': '', #path to save data
    'flare_intensity_threshold': 5e6, #threshold for flare intensity
}
print('global_constants:', global_constants)



# Set global styles: black background, white text.
# use plt.rcParams.update(plt.rcParamsDefault) to go back to default
plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams.update({
    'figure.facecolor': 'black',   # Figure background color
    'axes.facecolor': 'black',     # Axes background color
    'axes.edgecolor': 'white',     # Axes edge color
    'axes.labelcolor': 'white',    # X and Y labels
    'xtick.color': 'white',        # X-tick labels
    'ytick.color': 'white',        # Y-tick labels
    'grid.color': 'white',         # Gridlines color
    'text.color': 'white',         # All text (titles, etc.)
    'legend.frameon': True,        # Legend frame (turn it off or set color)
    'legend.facecolor': 'black',   # Legend background
    'legend.edgecolor': 'white',   # Legend edge
    'legend.framealpha': 1.0,      # Fully opaque
    'savefig.facecolor': 'black',  # Set saved figure background color to black
    'savefig.edgecolor': 'black',  # Set saved figure edge color to black
    # 'axes.spines.right': False,    # Disable right border
    # 'axes.spines.top': False,      # Disable top border
    # 'axes.spines.left': True,     # enable left border
    # 'axes.spines.bottom': True,   # enable bottom border
})
print('all figures will be black now.')
# plt.rcParams.update(plt.rcParamsDefault)




global_constants: {'fallback_colour': 'grey', 'global_triple_weight_factor': 3, 'global_save_path': '/sml/jannaschk/Progress_meetings/Progress_21_11/', 'flare_intensity_threshold': 5000000.0}
all figures will be black now.


In [2]:
# Basic Kmeans and centroid summary functions

def mini_batch_k_means(X, n_clusters=10, batch_size=1000, n_init=10, SOM = True, verbose=0):
    '''
    This method was taken from Brandon Panos and then modified to fit the needs of the project.

    Return centroids and labels of a data set using the k-means algorithm in minibatch style

    input   - X: data in the form [m_examples, lambda]
            - n_clusters: number of groups to find in the data
            - batch_size: number of data points used for a single update step of the centroids (these add together in a running stream)
            - n_init: number of convergences tested, the iteration with the lowest inertia is retained
            - SOM: If activated, will order the centroids in a self-organising map,
              which is an order that means centroids looking similarly are next to each other.
              ATTENTION: This will not relabel the labels. The goal is just to find the centroids,
              the labels will not match anymore.
            - verbose: output statistics
    output: - centroids: form [n_clusters, lambda], mean points of groups
            - labels: list of assignments indicating which centroid each data point belongs to
            - inertia: measure of performance, sum of all intercluster distances
            - n_clusters: number of clusters used
    '''
    mbk = MiniBatchKMeans(init='k-means++', n_clusters=n_clusters, batch_size=batch_size,
                            n_init=n_init, max_no_improvement=10, verbose=verbose)

    mbk.fit(X)

    centroids = mbk.cluster_centers_
    labels = mbk.labels_
    inertia = mbk.inertia_

    if SOM:
        # now, kmeans with the self-organising map
        centroids_som = som.obtain_som_ordered_data(centroids_to_be_ordered = centroids)
        return centroids_som, labels, inertia, n_clusters

    else:
        # just the usual kmeans, no additional ordering
        return centroids, labels, inertia, n_clusters


def assign_mg2k_centroids( X, centroids ):
    '''
    This function assigns the nearest centroid to each spectrum in X.
    In other words, it gives the labels for each spectrum in the data set X.
    
    in:
    - X: data in the form [m_examples, lambda]
    - centroids: form [n_clusters, lambda]
    out:
    - assigned_mg2k_centroids: list of labels indicating which centroid each spectrum belongs to
    '''

    # centroid_ids = list( range( centroids.shape[0] ) )
    centroid_ids = list(range(len(centroids)))


    # check whether X comes in the correct dimensions
    #if not X.shape[1] == centroids.shape[1]:
    #    raise ValueError( "Expecting X to have shape (_,{}). Please interpolate accordingly (More information with 'help(assign_mg2k_centroids)').".format( centroids.shape[1] ) )

    # create nearest centroid finder instance and fit it
    knc = NearestCentroid()
    knc.fit( X=centroids, y=centroid_ids )

    # predict nearest centroids for the supplied spectra
    # (making sure that X is normalized)
    assigned_mg2k_centroids = knc.predict( X ) # do we really need this normalize function here?

    # return vector of assigned centroids

    return assigned_mg2k_centroids 


def histogram_of_group_members_per_cluster(labels, n_clusters, top_var_indices, top_list_var, additional_title = ''):
    '''
    plots a histogram of the number of members per group
    input:  - labels: list of labels indicating which centroid each spcetrum belongs to
            - n_clusters: number of clusters used in the kmeans algorithm (has to match with the number of different labels)
            - top_var_indices: indices of the top variables
            - top_list_var: list of the top variables
            both of these vars are coming from the fct centroid_summary, so look there for more info
            - additional_title: additional title to be displayed
    result: - histogram of the number of members per group, but will only show the plot, not save it or return it.
    '''

    bins = np.arange(n_clusters + 1) - 0.5

    width = n_clusters * 25 / 80

    plt.figure(figsize=(width, 6))
    ax = sns.histplot(data=labels, bins = bins, color = '#2B35AF' ,fill = True, shrink = 0.5, element = 'bars')

    plt.xticks(range(n_clusters), [str(i) for i in range(n_clusters)])   

    heights_bars = [] 

    for patch in ax.patches:
        height = patch.get_height()
        heights_bars.append(height)

    index = 0
    for k in top_var_indices:
        plt.gca().get_xticklabels()[k].set_color('red')
        ax.patches[k].set_facecolor('red')

        position_k = plt.gca().get_xticklabels()[k].get_position()

        text = f'{top_list_var[index]:.1f}'
        #text = 'df'
        text_height = heights_bars[k] + 1000
        '''
        if text_height < 15000:
            text_height = text_height 
        '''

        ax.text(position_k[0], text_height, text, ha='center', color='red', fontsize=9)  # Replace 'max_value' with 'max_values'

        index += 1

    
    plt.title(f'# Group Members per Cluster: {additional_title}')
    plt.xlabel('Cluster #')

    plt.show()


def apply_centroids_to_obs(centroids,  obs_list: list, closest_mediods,
                           list_remove_mode = True,  list_groups_to_remove = [],
                           merge_mode = False, merge_dict = {},
                           plot_only_histogram = False, plot_no_histogram = False, verbose = 0,
                           skip_spectra = 0,
                           packed_subplots = False, plot_group_spectra_as_well = True, file_name_to_save = None):
    
    '''
    This fct is like the centroid_summary fct, but it applies the centroids to a list of obs_ids, and plots the results for each obs_id.
    So you can use it compare different obs, visually, or by the histogram.
    Almost all params are then used in the centroid_summary fct, so look there for more info.
    '''


    if list_remove_mode and merge_mode:
        print('\033[91mError: cannot use list_remove_mode and merge_mode at the same time.\033[0m')
        return None

    directory = '/sml/jannaschk/IRIS_30_obs_prepared'


    not_found_list = []
    matching_file_paths = []
    for search_string in obs_list:
        #print('search_string:', search_string)
        string_was_found = False
        for root, dirs, files in os.walk(directory):
            #print('hello')
            #print('root:', root)
            #print('dirs:', dirs)
            #print('files:', files)
            for dir in dirs:
                #for file in files:
                    #print(f'{file}')
                    #print(f'{dir}')
                    if search_string in dir:
                        matching_file_paths.append(os.path.join(root, dir))
                        string_was_found = True
                        #print(matching_file_paths)
        if not string_was_found:
            not_found_list.append(search_string)
    if len(matching_file_paths) == 0:
        print('No matching files found')
        return None
    for path in matching_file_paths:

        obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                                filename = f'{path[-26:]}',
                                                line = 'MgIIk',
                                                typ = f'{path[-29:-27]}')
        
        Data = obs_cls.im_arr_global[:,:,:].reshape(-1,960)
        Data = utils_kmeans.filter_out_zero_spectra(Data)
        labels = assign_mg2k_centroids(X = Data, centroids = centroids)

        centroid_summary(data = Data, centroids = centroids, 
                         labels = labels, closest_mediods = closest_mediods,
                         list_remove_mode = list_remove_mode, list_groups_to_remove = list_groups_to_remove,
                         additional_title = f'{path[-29:]}',
                         var_modus = 'above 2',
                         merge_mode = merge_mode, merge_dict = merge_dict,
                         plot_only_histogram = plot_only_histogram,
                         plot_group_spectra_as_well = plot_group_spectra_as_well,
                         plot_no_histogram = plot_no_histogram,
                         verbose = verbose,
                         skip_spectra = skip_spectra,
                         packed_subplots = packed_subplots,
                         file_name_to_save = file_name_to_save)
        print('')

    n_groups = len(centroids)
    # show groups that are not put into merged groups, ie leftover groups
    list_leftover_groups = []
    for centroid_number in range(n_groups):
        all_merged_groups = list(merge_dict.values())
        all_merged_groups = [item for sublist in all_merged_groups for item in sublist]
        if centroid_number not in all_merged_groups:
            list_leftover_groups.append(centroid_number)

    print(f'The merge_dict values are, ie all the groups that have been merged: {merge_dict.values()}')
    print(f'The following groups were not put into the merge_dict, and were not displayed as a consequence: {list_leftover_groups}')
    print('')

    if len(not_found_list) > 0:
        print(f'The following obs_ids were not found: {not_found_list}')
        print('')
    
    print('Application of centroids to obs list is done.')

    return None


def centroid_summary(data, centroids, labels, closest_mediods,
                     show_centroids_too = False,
                     additional_title ='', var_modus = 'above 2', 
                     merge_mode = False,
                     merge_dict = {},
                     list_remove_mode = True, 
                     list_groups_to_remove = [],
                     plot_only_histogram = False,
                     plot_group_spectra_as_well = False,
                     plot_no_histogram = False,
                     verbose = 0,
                     skip_spectra = 50,
                     packed_subplots = False,
                     max_cols = 4,
                     file_name_to_save = None,
                     path_to_save = None):
    '''
    plots a summary of the centroids found by the k-means run, plus a histogram of the number of members per group
    input:  - data: the data used for the k-means run
            - centroids: the centroids found by a k-means run
            - labels: list of labels indicating which centroid each spcetrum belongs to
            - closest_mediods: the closest real spectra to the centroids
            - additional_title: additional title to be displayed
            - var_modus: '75' or 'above 2', depending on the mode of the variance calculation (top 75 % or above 2 in total variance)
            - merge_mode: if activated, the centroids are merged into clumped groups, that are determined by the merge_dict
            - merge_dict: dictionary of the merged groups, which groups are merged together
            - list_remove_mode: if activated, the centroids are plotted without certain groups
            - list_groups_to_remove: list of groups to remove from this plot
            these two things are used when you want to merge groups, so you plot all groups, then start merging,
            and then you can remove the groups that are merged together from the overview, until you have merged everything
            - plot_only_histogram: if activated, only the histogram is plotted
            - plot_group_spectra_as_well: if activated, the group spectra are plotted as well,
              see skip_spectra for if you want to skip some spectra, or if you want to draw all spectra in that group.
              Gets computationally heavy and long fast though.
            - plot_no_histogram: if activated, only the centroids are plotted, and no histogram
            - verbose: output statistics:
              = 0 means no additional info on the plot, = 1 means additional info on the plot, like #spectra, variances, and more
            - skip_spectra: number of spectra to skip for the overplotting of the group spectra onto the mediods
            - packed_subplots: if activated, the subplots are packed together, so no space between them, and more nice to look at
            - max_cols: maximum number of columns for the subplots, depending on the window size of the computer
            - file_name_to_save: if a string is given, the plot is saved as a pdf.



    '''

    centroid_colour = 'red'

    if list_remove_mode and merge_mode:
        print('\033[91mError: cannot use list_remove_mode and merge_mode at the same time.\033[0m')
        return None

    n_groups = len(centroids)   # Number of groups

    if list_remove_mode:
        if len(list_groups_to_remove) > (n_groups - 1):
            print('Error: there must be at least two groups left. Now there are none or only one group left.')
            return None

    closest_real_spectra = closest_mediods


    if list_remove_mode:
        list_var = []
        for i in range(len(centroids)):
            cluster_data_i = data[labels == i]
            cluster_var = utils_kmeans.compute_cluster_var(cluster_data_i)
            list_var.append(cluster_var)

    if merge_mode:
        dict_vars = {}
        for key in merge_dict.keys():
            cluster_data = data[np.isin(labels, merge_dict[key])]
            cluster_var = utils_kmeans.compute_cluster_var(cluster_data)
            dict_vars[key] = cluster_var

    if list_remove_mode:
        if var_modus == '75':
            var_array = np.array(list_var)
            percentile_ = np.percentile(var_array, 75)
            top_elements = var_array[var_array >= percentile_]
            top_var_indices = np.where(var_array >= percentile_)[0]
            top_list_var = top_elements.tolist()
        elif var_modus == 'above 2':
            var_array = np.array(list_var)
            top_elements = var_array[var_array >= 2]
            top_var_indices = np.where(var_array >= 2)[0]
            top_list_var = top_elements.tolist()
        
        list_var = [round(var, 2) for var in list_var]
        top_list_var = [round(var, 2) for var in top_list_var]


    if merge_mode:
        all_cluster_vars = list(dict_vars.values())
        all_cluster_vars = np.array(all_cluster_vars)

        if var_modus == '75':
            percentile_ = np.percentile(all_cluster_vars, 75)
            top_elements = all_cluster_vars[all_cluster_vars >= percentile_]
            top_var_indices = np.where(var_array >= percentile_)[0]
            top_list_var = top_elements.tolist()
        elif var_modus == 'above 2':
            top_elements = all_cluster_vars[all_cluster_vars >= 2]
            top_var_indices = np.where(all_cluster_vars >= 2)[0]
            top_list_var = top_elements.tolist()

        top_dict_var = dict_vars.copy()
        for key, value in dict_vars.items():
            if value not in top_list_var:
                del top_dict_var[key]

        top_dict_var = {key: round(value, 2) for key, value in top_dict_var.items()}
        dict_vars = {key: round(value, 2) for key, value in dict_vars.items()}


    if plot_only_histogram:
        histogram_of_group_members_per_cluster(labels, n_clusters = n_groups, top_var_indices = top_var_indices,
                                               top_list_var = top_list_var, additional_title = additional_title)
        return None

    in_text = 10
    obs_wavelength=np.linspace(2794,2806,960)
    x_ticks = [2795, 2800, 2805]
    x_tick_labels = ['2795', '2800', '2805']
    num_centroids = len(centroids)

    num_centroids_to_display = num_centroids

    if packed_subplots:
        max_cols = max_cols + 1

    # Figure creation:
    #--------------------------------------------------------------------------------------------------------------------
    if list_remove_mode:
        num_centroids_to_display = num_centroids - len(list_groups_to_remove)
        cols = min(max_cols, num_centroids_to_display)  # Limit the number of columns to the max_cols or the number of centroids
        rows = (num_centroids_to_display + cols - 1) // cols  # Calculate the number of rows needed
        subplot_width_inches = 5
        subplot_height_inches = 2
        total_width_inches = cols * subplot_width_inches
        total_height_inches = rows * subplot_height_inches
        fig, axs = plt.subplots(rows, cols, figsize=(total_width_inches, total_height_inches),
                                gridspec_kw={'width_ratios': [subplot_width_inches]*cols,
                                            'height_ratios': [subplot_height_inches]*rows})
        ax = axs.ravel()
    
    elif merge_mode:
        num_centroids_to_display = len(merge_dict)
        cols = min(max_cols, num_centroids_to_display)  # Limit the number of columns to the max_cols or the number of centroids
        rows = (num_centroids_to_display + cols - 1) // cols  # Calculate the number of rows needed
        subplot_width_inches = 5
        subplot_height_inches = 2
        total_width_inches = cols * subplot_width_inches
        total_height_inches = rows * subplot_height_inches
        fig, axs = plt.subplots(rows, cols, figsize=(total_width_inches, total_height_inches),
                                gridspec_kw={'width_ratios': [subplot_width_inches]*cols,
                                            'height_ratios': [subplot_height_inches]*rows})
        ax = axs.ravel()
    
    #--------------------------------------------------------------------------------------------------------------------


    distinct_colors = sns.color_palette('cool', num_centroids_to_display).as_hex()  # Assuming n_groups is the same as num_centroids
    clr_dic = {i: color for i, color in enumerate(distinct_colors)}
    number_of_total_spectra = len(data)
    # plotting -----------------------------------------------------------------------------------------------

    if list_remove_mode:
        current_plot_index = 0  # To track the index for plotting
        for k in tqdm(range(num_centroids), desc='Plotting centroids'):
            if k in list_groups_to_remove:
                continue
            color_ = clr_dic.get(k, 'black')
            color_ = 'black'
            ax[current_plot_index].plot(obs_wavelength, closest_real_spectra[k], color=color_, linewidth=1, zorder=2)
            if show_centroids_too:        
                ax[current_plot_index].plot(obs_wavelength, centroids[k], color= centroid_colour, linewidth=1, zorder=2)

            cluster_data_k = data[labels == k]
            number_of_members_k = len(cluster_data_k)
            if plot_group_spectra_as_well and verbose == 1:
                cluster_data_k_reduced = cluster_data_k[::skip_spectra]  # Select every nth spectrum for plotting
                ax[current_plot_index].plot(obs_wavelength, cluster_data_k_reduced.T, color='#bbbbbb', alpha=0.4, zorder=1)  # Transpose the array to align dimensions for plotting
            ax[current_plot_index].text(0.01, 0.85, str(k), transform=ax[current_plot_index].transAxes, size=in_text, fontweight='bold')
            if verbose == 1:
                if list_var[k] in top_list_var:
                    ax[current_plot_index].text(0.2, 0.85, f'VAR: {list_var[k]}', transform=ax[current_plot_index].transAxes, size=in_text, fontweight='bold', color='red')
                else:
                    ax[current_plot_index].text(0.2, 0.85, f'VAR: {list_var[k]}', transform=ax[current_plot_index].transAxes, size=in_text)
                ax[current_plot_index].text(0.5, 0.85, f'#spec: {number_of_members_k} = ', transform=ax[current_plot_index].transAxes, size=in_text)
                ax[current_plot_index].text(0.85, 0.85, f'{round(number_of_members_k / number_of_total_spectra * 100, 1)} %', transform=ax[current_plot_index].transAxes, size=in_text)

            ax[current_plot_index].set_ylim(-0.1, 1.3)
            ax[current_plot_index].set_xticks(x_ticks)
            ax[current_plot_index].set_xticklabels(x_tick_labels)

            if packed_subplots:
                ax[current_plot_index].set_xticks([])
                ax[current_plot_index].set_yticks([])

            if verbose == 1 and len(cluster_data_k) == 0:
                ax[current_plot_index].set_facecolor("#eeeeee")
            current_plot_index += 1


        for k in range(current_plot_index, len(ax)):
            fig.delaxes(ax[k])
        
        fig.subplots_adjust(hspace=0.5, wspace=0.2)
        
        if packed_subplots:
            fig.subplots_adjust(hspace=0.0, wspace=0.0)

        add_on_title = ''
        if plot_group_spectra_as_well:
            add_on_title = f'every {skip_spectra}th spectrum overplotted'

        title_text_size = 20
        fig.text(0.5, 0.91, f'Mg II h&k mediods, {additional_title}, {add_on_title}', ha='center', va='center', rotation='horizontal',size=title_text_size)
        fig.text(0.5, 0.05, 'Wavelength [$\AA$]', ha='center', va='center', rotation='horizontal',size=title_text_size)
        fig.text(0.05, 0.5, 'Normalized intensity', ha='center', va='center', rotation='vertical',size=title_text_size)

        if not file_name_to_save is None and not path_to_save is None:
            # plt.savefig(f'{path_to_save}{file_name_to_save}.png')
            save_plot(title = file_name_to_save, format = 'png')
            save_plot(title = file_name_to_save, format = 'pdf')




        plt.show()

    elif merge_mode:
        combined_data_dict = {key: data[np.isin(labels, merge_dict[key])] for key in merge_dict.keys()}
        current_plot_index = 0
        for key, value in merge_dict.items():

            combined_data = combined_data_dict[key]

            #print(f'the key is currently: {key}')

            for centroid_number in merge_dict[key]: # plot the different centroids that are in the same merged group
                ax[current_plot_index].plot(obs_wavelength, closest_real_spectra[centroid_number], linewidth=1, zorder=2, label = f'{centroid_number}')
                
            number_of_members_k = len(combined_data)

            if plot_group_spectra_as_well and verbose == 1:
                combined_data_reduced = combined_data[::skip_spectra]
                ax[current_plot_index].plot(obs_wavelength, combined_data_reduced.T, color='#bbbbbb', alpha=0.4, zorder=1)  # Transpose the array to align dimensions for plotting

            legend_fontsize = 10
            if len(np.array(value)) > 6:
                legend_fontsize = 8

            ax[current_plot_index].text(0.01, 0.85, str(key), transform=ax[current_plot_index].transAxes, size=in_text, fontweight='bold')
            ax[current_plot_index].legend(loc='upper right', fontsize=legend_fontsize, handletextpad=0.5, columnspacing=0.3, labelspacing=0.05)

            if verbose == 1:
                if key in top_dict_var.keys():
                    ax[current_plot_index].text(0.2, 0.85, f'VAR: {top_dict_var[key]}', transform=ax[current_plot_index].transAxes, size=in_text, color='red', fontweight = 'bold')
                else:
                    ax[current_plot_index].text(0.2, 0.85, f'VAR: {dict_vars[key]}', transform=ax[current_plot_index].transAxes, size=in_text)
                ax[current_plot_index].text(0.5, 0.85, f'#spec: {number_of_members_k} = ', transform=ax[current_plot_index].transAxes, size=in_text)
                ax[current_plot_index].text(0.85, 0.85, f'{round(number_of_members_k / number_of_total_spectra * 100, 1)} %', transform=ax[current_plot_index].transAxes, size=in_text)

            ax[current_plot_index].set_ylim(-0.1, 1.3)
            ax[current_plot_index].set_xticks(x_ticks)
            ax[current_plot_index].set_xticklabels(x_tick_labels)
            if packed_subplots:
                ax[current_plot_index].set_xticks([])
                ax[current_plot_index].set_yticks([])
            if verbose == 1 and len(combined_data) == 0:
                ax[current_plot_index].set_facecolor("#eeeeee")
            current_plot_index += 1


        for k in range(current_plot_index, len(ax)):
            fig.delaxes(ax[k])

        fig.subplots_adjust(hspace=0.5, wspace=0.2)
        if packed_subplots:
            fig.subplots_adjust(hspace=0.0, wspace=0.0)

        add_on_title = ''
        if plot_group_spectra_as_well:
            add_on_title = f'every {skip_spectra}th spectrum overplotted'

        title_text_size = 15
        fig.text(0.5, 0.95, f'Mg II h&k mediods, {additional_title}, {add_on_title}', ha='center', va='center', rotation='horizontal',size=title_text_size)
        fig.text(0.5, 0, 'Wavelength [$\AA$]', ha='center', va='center', rotation='horizontal',size=title_text_size)
        fig.text(0.05, 0.5, 'Normalized intensity', ha='center', va='center', rotation='vertical',size=title_text_size)


        if not file_name_to_save is None and not path_to_save is None:
            # plt.savefig(f'{path_to_save}{file_name_to_save}.png')
            save_plot(title = file_name_to_save, format = 'png')
            save_plot(title = file_name_to_save, format = 'pdf')


        
        plt.show()



    if not (merge_mode or list_remove_mode):
        if plot_no_histogram:
            return None

        histogram_of_group_members_per_cluster(labels, n_clusters=n_groups, top_var_indices=top_var_indices, top_list_var=top_list_var, additional_title=additional_title)
        return None

    return None


def save_plot(title, format):
    '''
    Saves the current plot with the given title and format, ensuring the legend is included.
    '''
    save_path = global_constants['global_save_path']

    # Check if the file already exists and add a counter if it does
    counter = 1
    original_title = title

    while os.path.exists(f"{save_path}{title}.{format}"):
        title = f"{original_title}_{counter}"
        counter += 1

    # Save the plot with bbox_inches='tight' to include the legend
    plt.savefig(f"{save_path}{title}.{format}", format=format, bbox_inches='tight', pad_inches=0.1)

    return None


In [3]:
# ---- triplet modification ----


def weigh_the_triplet(spectral_data, triplet_weight_factor, show_plots = False, verbose = 0):
    '''
    returns the weighted spectra:
    the triplet is enhanced by a factor of triplet_weigh_factor
    If there is no emission in the triplet, then don't enhance it,
    so it doesn't change the spectrum.
    '''

    if spectral_data.shape[0] > 500:
        show_plots = False

    def is_there_triplet_emission(spectrum):
        '''
        returns True if there is emission in the triplet in that spectrum, False otherwise.
        '''
        wavelengths = np.linspace(2794,2806,960)

        lowest_triplet = 2798.6
        middle_triplet = 2798.75
        highest_triplet = 2798.82
        spectrum_value_at_lowest_triplet = spectrum[np.abs(wavelengths - lowest_triplet).argmin()]
        spectrum_value_at_middle_triplet = spectrum[np.abs(wavelengths - middle_triplet).argmin()]
        spectrum_value_at_highest_triplet = spectrum[np.abs(wavelengths - highest_triplet).argmin()]

        triplet_values = [spectrum_value_at_lowest_triplet, spectrum_value_at_middle_triplet, spectrum_value_at_highest_triplet]
        max_triplet_index = np.argmax(triplet_values)
        triplet_wavelength = [lowest_triplet, middle_triplet, highest_triplet][max_triplet_index]

        closest_index_triplet = np.abs(wavelengths - triplet_wavelength).argmin()
        #print('Closest index to triplet:', closest_index_triplet)
        spectrum_value_at_triplet = spectrum[closest_index_triplet]
        #print('spectrum value at triplet:', spectrum_value_at_triplet)

        #print('spectrum:', spectrum)
        #print('spectrum shape:', spectrum.shape)

        background_indices = np.where((wavelengths >= 2798) & (wavelengths <= 2802))

        #emission_vs_background = np.log10( spectrum_value_at_triplet / np.mean(spectrum[background_indices]) )
        #return (emission_vs_background > 0)

        emission_vs_background =  spectrum_value_at_triplet / np.mean(spectrum[background_indices]) 

        return (emission_vs_background > 1.9) #1.9 for now. 1 was jonas initial suggestion.

    wavelengths = np.linspace(2794,2806,960)
    lowest_triplet = 2798.6
    middle_triplet = 2798.75
    highest_triplet = 2798.82
    margin = 0.1 #how much margin right of highest triple to still weigh it.
    triplet_region_indices = np.where( (wavelengths >= lowest_triplet - margin) &
                                      (wavelengths <= highest_triplet + margin) )[0]
    
    triplet_emission_counter = 0

    spectral_data_copy = spectral_data.copy()

    for spectrum in spectral_data_copy:
        if is_there_triplet_emission(spectrum):
            #print('triplet region indices:', triplet_region_indices)
            original_spectrum = spectrum.copy()
            spectrum[triplet_region_indices] *= triplet_weight_factor
            triplet_emission_counter += 1

            # plot:
            if show_plots:
                fig, axs = plt.subplots(1, 2, figsize=(8, 4))

                # Plot the original spectrum
                axs[0].plot(wavelengths, original_spectrum, color = 'black')
                axs[0].set_title('Original Spectrum')
                axs[0].axvline(x=middle_triplet, color='red', linestyle='--')
                #axs[0].set_xlabel('Wavelength')
                #axs[0].set_ylabel('Intensity')
                #axs[0].legend()

                # Plot the weighted spectrum
                axs[1].plot(wavelengths, spectrum, color = 'darkorchid')
                axs[1].axvline(x=middle_triplet, color='red', linestyle='--')
                axs[1].set_title('Weighted Spectrum')
                #axs[1].set_xlabel('Wavelength')
                #axs[1].set_ylabel('Intensity')
                #axs[1].legend()

                plt.tight_layout()
                #plt.show()

    if verbose  == 1:
        print('triplet_emission_counter:', triplet_emission_counter)
        print('total number of spectra:', spectral_data_copy.shape[0])
        percentage = np.round(triplet_emission_counter / spectral_data_copy.shape[0] * 100, 1)
        print(f'percentage of spectra with triplet emission: {percentage} %')
    if show_plots:
        plt.show()

    
    return spectral_data_copy 


def unweigh_the_triplet_again(spectral_data, triplet_weight_factor):
    '''
    reverses the weighing of the triplet.
    Used for centroids and getting back the original data.
    '''
    if triplet_weight_factor == 0:
        print('triplet_weight_factor is 0, so no unweighing /reversing can be done.')
        return None

    def is_there_triplet_emission(spectrum):
        '''
        returns True if there is emission in the triplet in that spectrum, False otherwise.
        '''
        wavelengths = np.linspace(2794,2806,960)
        lowest_triplet = 2798.6
        middle_triplet = 2798.75
        highest_triplet = 2798.82
        spectrum_value_at_lowest_triplet = spectrum[np.abs(wavelengths - lowest_triplet).argmin()]
        spectrum_value_at_middle_triplet = spectrum[np.abs(wavelengths - middle_triplet).argmin()]
        spectrum_value_at_highest_triplet = spectrum[np.abs(wavelengths - highest_triplet).argmin()]
        triplet_values = [spectrum_value_at_lowest_triplet, spectrum_value_at_middle_triplet, spectrum_value_at_highest_triplet]
        max_triplet_index = np.argmax(triplet_values)
        triplet_wavelength = [lowest_triplet, middle_triplet, highest_triplet][max_triplet_index]
        closest_index_triplet = np.abs(wavelengths - triplet_wavelength).argmin()
        spectrum_value_at_triplet = spectrum[closest_index_triplet]
        background_indices = np.where((wavelengths >= 2798) & (wavelengths <= 2802))
        emission_vs_background =  spectrum_value_at_triplet / np.mean(spectrum[background_indices]) 
        return (emission_vs_background > 1.9) #1.9 for now. 1 was jonas initial suggestion.

    wavelengths = np.linspace(2794,2806,960)
    lowest_triplet = 2798.6
    middle_triplet = 2798.75
    highest_triplet = 2798.82
    margin = 0.1 #how much margin right of highest triple to still weigh it.
    triplet_region_indices = np.where( (wavelengths >= lowest_triplet - margin) &
                                      (wavelengths <= highest_triplet + margin) )[0]
    
    spectral_data_copy = spectral_data.copy()
    for spectrum in spectral_data_copy:
        if is_there_triplet_emission(spectrum):
            # here is the only change to the original weighing fct: divide by the triplet_weight_factor
            spectrum[triplet_region_indices] *= (1 / triplet_weight_factor)
    return spectral_data_copy 


def subsample_kmeans_on_a_group(group_to_subsample, weighed_centroids, obs_id, n_subclusters = 2):

    '''
    this fct will use a single group of the kmeans clustering, and subsample it.
    The goal is to do it on a group where there is still a high variance and is doesn't visually look good.
    So, it might mean there are more single groups hiding in that group.
    
    The fct does:
    - takes a group
    - weighs it, so the triplet is enhanced
    - subsamples it on n_clusters (2, 3, or 4 clusters)
    - Then shows those new groups in a plot, using list_remove_mode of the centroid_summary fct.
    Then you can decide if it looks good or not.

    returns:
    - the new (weighed) centroids.
    '''


    # get the spectral data for that obs_id:
    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'PF')
    Data = obs_cls.im_arr_global[:,:,:].reshape(-1,960)
    total_spectra_data_unweighed = utils_kmeans.filter_out_zero_spectra(Data)

    # find the spectra of only that group:
    total_spectra_data_weighed = weigh_the_triplet(total_spectra_data_unweighed, global_constants['global_triple_weight_factor'])
    labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = weighed_centroids)
    spectral_data_of_that_group = total_spectra_data_weighed[labels == group_to_subsample]

    # weigh the data for the triplet:
    spectral_data_of_that_group_weighed = weigh_the_triplet(spectral_data_of_that_group, global_constants['global_triple_weight_factor'])

    # run kmeans on those spectra of that group:
    subsampled_centroids_weighed, _, subsampled_inertia, _ = mini_batch_k_means(X = spectral_data_of_that_group_weighed,
                                            n_clusters=n_subclusters,
                                            batch_size=1000,
                                            n_init=10, SOM = True, verbose=0)

    # now use the centroid_summary fct to plot the new subgroups:
    # but first, redo the centroids_list and redo the labels of course:

    # centroids:
    new_weighed_centroids = weighed_centroids.copy()
    list_1 = new_weighed_centroids.tolist()
    list_1.pop(group_to_subsample)
    list_1.extend(subsampled_centroids_weighed)
    new_weighed_centroids = np.array(list_1)
    new_unweighed_centroids = unweigh_the_triplet_again(new_weighed_centroids, global_constants['global_triple_weight_factor'])
    #labels:
    new_labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = new_weighed_centroids)
    # mediods:
    closest_mediods = utils_kmeans.find_closest_mediods(centroids = new_weighed_centroids, Data = total_spectra_data_weighed)
    closest_mediods = unweigh_the_triplet_again(closest_mediods, global_constants['global_triple_weight_factor'])

    #now find the list of the groups we don't want to see: (we only want to see the new subgroups)
    list_groups_to_remove = list(range(len(new_unweighed_centroids)))
    # keep only the last new subclusters: the list contains all the group that we don't want to see.
    list_groups_to_remove = list_groups_to_remove[:-n_subclusters]


    centroid_summary(data = total_spectra_data_unweighed,
                     centroids = new_unweighed_centroids,
                     labels = new_labels,
                     closest_mediods = closest_mediods,
                     show_centroids_too = True,
                     additional_title = f' subsampled group: {group_to_subsample}', var_modus = 'above 2', 
                        merge_mode = False,
                        merge_dict = {},
                        list_remove_mode = True, 
                        list_groups_to_remove = list_groups_to_remove,
                        plot_only_histogram = False,
                        plot_group_spectra_as_well = True,
                        plot_no_histogram = True,
                        verbose = 1,
                        skip_spectra = 1,
                        packed_subplots = True,
                        max_cols = 4,
                        file_name_to_save = None,
                        path_to_save = None)



    return new_weighed_centroids, subsampled_inertia


def subsample_group_with_different_n_clusters(group_to_subsample, weighed_centroids,
                                              obs_id, n_subcluster_list, path_to_save = None):
    '''
    does the same as the fct subsample_kmeans_on_a_group, but here,
    you can choose to repeat it with different n_subclusters.

    It then shows all the relevant plots.

    n_subcluster_list is a list of integers, e.g. [2, 3, 4, 5], for the numbers you want to show plots for
    For example do [2, 5, 10, 15].
    '''


    # get the spectral data for that obs_id:
    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'PF')
    Data = obs_cls.im_arr_global[:,:,:].reshape(-1,960)
    total_spectra_data_unweighed = utils_kmeans.filter_out_zero_spectra(Data)

    # find the spectra of only that group:
    total_spectra_data_weighed = weigh_the_triplet(total_spectra_data_unweighed, global_constants['global_triple_weight_factor'])
    labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = weighed_centroids)
    spectral_data_of_that_group = total_spectra_data_weighed[labels == group_to_subsample]

    # weigh the data for the triplet:
    spectral_data_of_that_group_weighed = weigh_the_triplet(spectral_data_of_that_group, global_constants['global_triple_weight_factor'])

    #here this fct does a loop instead:
    dict_of_centroids_for_certain_subclusters = {}

    for n_subclusters in n_subcluster_list:
        # run kmeans on those spectra of that group:
        subsampled_centroids_weighed, _, _, _ = mini_batch_k_means(X = spectral_data_of_that_group_weighed,
                                                n_clusters=n_subclusters,
                                                batch_size=1000,
                                                n_init=10, SOM = True, verbose=0)
        # now use the centroid_summary fct to plot the new subgroups:
        # but first, redo the centroids_list and redo the labels of course:
        # centroids:
        new_weighed_centroids = weighed_centroids.copy()
        list_1 = new_weighed_centroids.tolist()
        list_1.pop(group_to_subsample)
        list_1.extend(subsampled_centroids_weighed)
        new_weighed_centroids = np.array(list_1)

        # save those new centroids in a dict:
        dict_of_centroids_for_certain_subclusters[n_subclusters] = new_weighed_centroids
        #
        new_unweighed_centroids = unweigh_the_triplet_again(new_weighed_centroids, global_constants['global_triple_weight_factor'])
        #labels:
        new_labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = new_weighed_centroids)
        # mediods:
        closest_mediods = utils_kmeans.find_closest_mediods(centroids = new_weighed_centroids, Data = total_spectra_data_weighed)
        closest_mediods = unweigh_the_triplet_again(closest_mediods, global_constants['global_triple_weight_factor'])
        #now find the list of the groups we don't want to see: (we only want to see the new subgroups)
        list_groups_to_remove = list(range(len(new_unweighed_centroids)))
        # keep only the last new subclusters: the list contains all the group that we don't want to see.
        list_groups_to_remove = list_groups_to_remove[:-n_subclusters]
        # plot:
        centroid_summary(data = total_spectra_data_unweighed,
                        centroids = new_unweighed_centroids,
                        labels = new_labels,
                        closest_mediods = closest_mediods,
                        show_centroids_too = True,
                        additional_title = f' subsampled group: {group_to_subsample}, #subgroups: {n_subclusters}', var_modus = 'above 2', 
                        merge_mode = False,
                        merge_dict = {},
                        list_remove_mode = True, 
                        list_groups_to_remove = list_groups_to_remove,
                        plot_only_histogram = False,
                        plot_group_spectra_as_well = True,
                        plot_no_histogram = True,
                        verbose = 1,
                        skip_spectra = 1,
                        packed_subplots = True,
                        max_cols = 3,
                        file_name_to_save = f'subsampled_group_{group_to_subsample}_n_subclusters_{n_subclusters}_obsid_{obs_id}',
                        path_to_save = path_to_save)
        

    return dict_of_centroids_for_certain_subclusters



In [4]:
# animated centroid mask fct and other visualisation fcts:

def animate_mask(obs, centroids, clr_dic, start, end, fallback_colour = 'grey', interval_ms = 100):
    '''
    This method is taken from Brandon Panos
    '''

    """
    Returns an animation of the observation with a centroid mask overlay

    (slit will be coloured differently each time step, depending on the colour dictionary, in the background the sji Si image is shown)

    Parameters
    ----------
        obs : irisreader.observation
        centroids : float
            centroids found by k-means [n_centroids, lambda]
        clr_dic : {int:str}
            dictionary to color code each centroid 
        start : float in unix time
            start of the animated movie
        end : float in unix time
            end of the animated movie
        fallback_colour : str
            colour to use if a specific spectrum has no assigned colour in clr_dic
        interval_ms : int
            interval between frames in milliseconds
    """
   
    sji = obs.sji[0]  #index 1 means the Mg II h/k line, which is what we want or not ????. Index 0 would be the Si IV line.
    #times_in_unix = np.array( obs.sji[1].get_timestamps() )
    times_in_unix = obs.sji[0].get_timestamps()
    times_in_unix = np.array(times_in_unix)
    start_index = np.abs(times_in_unix - start).argmin()
    end_index =   np.abs(times_in_unix - end).argmin()

    
    # Here, we cut it to the desired length
    sji.cut(start_index, end_index)



    raster = obs.raster("Mg II k")
    raster_inds = [find_closest_raster( raster, sji, i )[0] for i in range(sji.n_steps)]    
    #---------------------------- label data ---------------------------
    labels = []
    for i in raster_inds:
        X = raster.get_interpolated_image_step( 
                    step = i, 
                    lambda_min = 2794,
                    lambda_max = 2806,
                    n_breaks = 960 #used to be 216 originally
        )
        labels.append( assign_mg2k_centroids( X, centroids ) )

    #---------------------------- set up animation ---------------------------
    #n = sji.shape[0]
    N = sji.shape[1]
    gamma = 0.1
    # initialize plot
    image = sji.get_image_step( 0 ).clip(min=0.01)**gamma
    slit_x_position = sji.get_slit_pos( 0 )
    slit_y_position = np.linspace( 0, image.shape[0]-1, N, dtype=np.int )
    sji_flare_mask=labels[0][slit_y_position]
    #print('The flare mask is:', sji_flare_mask)
    clr_mask = [clr_dic.get(str(sji_flare_mask[i]), fallback_colour) for i in range(len(sji_flare_mask))] # this reads the colours for each label,
    # and if the label was not in the colours dict, then it will be coloured grey in the video
    # however, I will try to use no colour (RGBA) (0, 0, 0, 0)


    #print('The colour mask is:', clr_mask)
    fig = plt.figure( figsize=(12,12) )
    im = plt.imshow( -image, cmap="Greys", origin='lower' ) #Greys gist_heat
    scat = plt.scatter( [slit_x_position]*N, slit_y_position, c=clr_mask, marker='D', s=10, alpha=0.2) #alpha=1 was orignal, s = 10

    # i want to display in the legend all the colours that are in the labels.

    # Convert to a set to remove duplicates
    groups_present_in_this_obs = list(set(np.concatenate(labels)))



    # do nothing in the initialization function
    def init():
        return im, scat
    # animation function
    def animate(i):
        slit_x_position = sji.get_slit_pos( i )
        sji_flare_mask=labels[i][slit_y_position]
        #xcenix = sji.headers[i]['XCENIX']
        #ycenix = sji.headers[i]['YCENIX']
        date_obs = sji.headers[i]['DATE_OBS']
        im.axes.set_title( "Frame {}: {}".format( i, date_obs ) )
        im.set_data( -sji.get_image_step( i ).clip(min=0.01)**gamma )

        legend_elements = []
        for group in groups_present_in_this_obs:
            legend_elements.append(Patch(facecolor=clr_dic.get(str(group), fallback_colour), edgecolor=clr_dic.get(str(group), fallback_colour), label=f'{group}') )
        # Add the legend to the axes
        im.axes.legend(handles=legend_elements, loc='upper right', title = '              Group')  # Adjust location as needed
        #im.axes.legend.get_title().set_fontsize('large')  # Optional: Set the title font size

        
        scat.set_offsets( np.array([ [slit_x_position]*N, slit_y_position ] ).T ) #this is setting the position of the squares
        #scat.set_array( sji_flare_mask )
        clr_mask = [clr_dic.get(str(sji_flare_mask[i]), fallback_colour) for i in range(len(sji_flare_mask))]
        scat.set_color( clr_mask )
        # it works like expected, if you comment the set_array line.
        return im, scat

    # Call the animator.  blit=True means only re-draw the parts that have changed.
    anim = animation.FuncAnimation(fig, animate, init_func=init, frames=sji.n_steps, interval=interval_ms, blit=True)
    # Close the plot
    plt.close(anim._fig)
    # Show animation in notebook
    return HTML(anim.to_html5_video()), anim


def evolving_slit_plot(skip_lines, obs_id,
                       centroids, clr_dic,
                       start, end, triplet_weight_factor, fallback_colour='grey'
                       #give_spectral_data = False, spectral_data = None
                       ):
    """
    Plots colored lines next to each other on a graph with time on the x-axis.
    It is like the result of the animate_mask fct, but no video. instead we see how the slit evolves through time.
    
    Parameters
    ----------
        obs_id : obs id.
        centroids : float
            centroids found by k-means [n_centroids, lambda]
        clr_dic : {int:str}
            dictionary to color code each centroid 
        start : float in HH:MM time
            start of the plot
        end : float in HH:MM time
            end of the plot
        fallback_colour : str
            colour to use if a specific spectrum has no assigned colour in clr_dic
    """

    year = obs_id[:4]
    month = obs_id[4:6]
    day = obs_id[6:8]
    pth = f'/sml/iris/{year}/{month}/{day}/{obs_id}'
    obs = observation( pth, keep_null=True )


    start_time_HH_MM = start
    end_time_HH_MM = end
    #print('the start time in HH:MM is:', start_time_HH_MM)
    #print('the end time in HH:MM is:', end_time_HH_MM)


    # time convert from 17:45 (HH:MM) to datetime object: ------------------------------
    # Convert start_time and end_time to Unix time using the date from obs_id
    date_str = obs_id[:8]  # Extract the date part from obs_id
    date_format = "%Y%m%d"  # Define the date format
    start_datetime_str = f"{date_str} {start}"
    end_datetime_str = f"{date_str} {end}"
    datetime_format = "%Y%m%d %H:%M"
    start_datetime = datetime.strptime(start_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    end_datetime = datetime.strptime(end_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    start_unix_time = int(start_datetime.timestamp())
    end_unix_time = int(end_datetime.timestamp())
    start = start_unix_time
    #print('the start time in unix time is:', start)
    end = end_unix_time
    #print('the end time in unix time is:', end)



    sji = obs.sji[0]  # Using the first SJI channel
    times_in_unix = np.array(sji.get_timestamps())
    #print('start and end of obs is:', [obs.start_date, obs.end_date])

    # Get indices for the start and end times
    start_index = np.abs(times_in_unix - start).argmin()
    end_index = np.abs(times_in_unix - end).argmin()

    # Cut to the desired length
    sji.cut(start_index, end_index)

    # Get raster data and assign centroids
    raster = obs.raster("Mg II k")
    raster_inds = [find_closest_raster(raster, sji, i)[0] for i in range(sji.n_steps)]
    
    labels = []
    for i in raster_inds:
        X = raster.get_interpolated_image_step(
            step=i,
            lambda_min=2794,
            lambda_max=2806,
            n_breaks=960  # Change as needed
        )
        X_weighed = weigh_the_triplet(X, triplet_weight_factor)
        labels.append(assign_mg2k_centroids(X_weighed, centroids))


    # Set up plot for all lines over time
    fig, ax = plt.subplots(figsize=(12, 4))
    N = sji.shape[1]
    slit_y_position = np.linspace(0, N-1, N, dtype=np.int)

    
    # Get only the time values for the steps that are actually plotted
    #times_to_plot = times_in_unix
    times_in_unix = np.array(sji.get_timestamps())
    times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
    if skip_lines:
        times_to_plot = times_to_plot[0::2]

    step = 0
    # Plot each time step as a line on the graph
    for i in range(sji.n_steps - 1):
        if i % 2 == 1 and skip_lines:  # Only plot every second line if skip_lines is True
            continue  # Skip every second step
        # Retrieve the flare mask for this step
        sji_flare_mask = labels[i][slit_y_position]
        # Color the line based on the flare mask
        clr_mask = [clr_dic.get(str(sji_flare_mask[j]), fallback_colour) for j in range(len(sji_flare_mask))]
        # Plot line with colors at each time step
        for j in range(len(slit_y_position) - 1):
            #print('step is:', step)
            #print('i is:   ', i)
            #print('times to plot at step:', times_to_plot[step])
            #print('times to plot at step + 1:', times_to_plot[step+1])
            ax.plot([times_to_plot[step], times_to_plot[step+1]], [slit_y_position[j], slit_y_position[j+1]],
                    color=clr_mask[j], marker='s', markersize=1, linestyle = '-', alpha=1)
        step += 1  # Increment the step each time a line is drawn


    # try doing it using imshow: not now.


    # Add labels, title, and legend
    ax.set_xlabel('Time')
    ax.set_ylabel('Spatial Position')
    ax.set_title('Colored Lines for Each Time Step')

    # Create a legend using the available groups
    groups_present_in_this_obs = list(set(np.concatenate(labels)))
    legend_elements = [Patch(facecolor=clr_dic.get(str(group), fallback_colour),
                             edgecolor=clr_dic.get(str(group), fallback_colour),
                             label=f'{group}') for group in groups_present_in_this_obs]
    ax.legend(handles=legend_elements, title='  Group', bbox_to_anchor=(1, 1))

    ax.xaxis.set_major_locator(MaxNLocator(nbins=20))
    # Format the x-axis to show time as HH:MM
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))



    plt.show()


def evolv_plot_with_only_subgroups(group_to_subsample, n_subclusters,
                                   start_hh_mm, end_hh_mm,
                                   obs_id, weighed_centroids):

    # do kmeans
    # do evolv plot with only those groups.
    # all different colours.

    # get the spectral data for that obs_id:
    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'PF')
    Data = obs_cls.im_arr_global[:,:,:].reshape(-1,960)
    total_spectra_data_unweighed = utils_kmeans.filter_out_zero_spectra(Data)
    # find the spectra of only that group:
    total_spectra_data_weighed = weigh_the_triplet(total_spectra_data_unweighed, global_constants['global_triple_weight_factor'])
    labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = weighed_centroids)
    spectral_data_of_that_group = total_spectra_data_weighed[labels == group_to_subsample]
    # weigh the data for the triplet:
    spectral_data_of_that_group_weighed = weigh_the_triplet(spectral_data_of_that_group, global_constants['global_triple_weight_factor'])
    # run kmeans on those spectra of that group:
    subsampled_centroids_weighed, _, subsampled_inertia, _ = mini_batch_k_means(X = spectral_data_of_that_group_weighed,
                                            n_clusters=n_subclusters,
                                            batch_size=1000,
                                            n_init=10, SOM = False, verbose=0)

    # but first, redo the centroids_list:
    new_weighed_centroids = weighed_centroids.copy()
    list_1 = new_weighed_centroids.tolist()
    list_1.pop(group_to_subsample)
    list_1.extend(subsampled_centroids_weighed)
    new_weighed_centroids = np.array(list_1)
    new_unweighed_centroids = unweigh_the_triplet_again(new_weighed_centroids, global_constants['global_triple_weight_factor'])

    # now only use the spectra of that group to be plotted
    # into different colours according to the subgroups

    def get_color_dict(array_of_subgroup_numbers, array_of_all_group_numbers):
        """
        Returns a dictionary where the keys are integers (0, 1, 2, ...) and the values are distinct colors.
        Parameters:
        n (int): Number of colors to generate
        Returns:
        dict: A dictionary where keys are integers and values are color strings.
        """
        # Get the default color cycle from matplotlib
        color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
        # If n exceeds the length of the color cycle, use the default colors repeatedly
        color_dict = {str(i): color_cycle[(i - len(array_of_all_group_numbers) + len(array_of_subgroup_numbers) + 2) % len(color_cycle)]
                      for i in array_of_subgroup_numbers}
        # fill up the rest with grey:
        for i in array_of_all_group_numbers:
            if str(i) not in color_dict:
                color_dict[str(i)] = 'grey'

        #special correction
        #color_dict['151'] = 'grey'

        return color_dict

    len_centr_before_subsampling = len(weighed_centroids)
    len_centr_after_subsampling = len(new_weighed_centroids)
    #print('The number of centroids before subsampling:', len_centr_before_subsampling)
    #print('The number of centroids after subsampling:', len_centr_after_subsampling)
    
    subgroups_numbers = list(range(len_centr_before_subsampling - 1, len_centr_after_subsampling))
    print('subgroups numbers:', subgroups_numbers)
    all_group_numbers = list(range(len_centr_after_subsampling))
    print('all group numbers:', all_group_numbers)
    
    clr_dic = get_color_dict(subgroups_numbers, all_group_numbers)
    print('colour dictionary:', clr_dic)
    #return None
    start = start_hh_mm
    end = end_hh_mm
    print('it arrived here.')
    # now plot the evolving slit plot with only those subgroups:
    evolving_slit_plot(skip_lines = False,
                       #give_spectral_data = False,
                       #spectral_data = None,
                       obs_id = obs_id,
                       centroids = new_weighed_centroids,
                       clr_dic = clr_dic,
                       start = start, end = end,
                       triplet_weight_factor = global_constants['global_triple_weight_factor'],
                       fallback_colour='grey')


    return None


def show_box_of_slit_evolution(obs_id, skip_spectra, merge_dict, clr_dic,
                               centroids, mediods,
                               start_time, end_time, box_limits_y, figure_save_name):
    

    '''
    fct that shows all the spectra that are in a selected part (box) of the evolving slit plot of the fct evolving_slit_plot fct

    It is used to show the spectra that are in a certain part of the slit, and make more detailed visual analysis.

    Apart from that, the fct works similarly to the centroid_summary fct, but has less output statistics (var, etc.)

    Also the default fallback colour of the clr_dic is grey.
    So all grey subplots are plotted last.

    '''


    lower_box_limit = box_limits_y[0]
    upper_box_limit = box_limits_y[1]

    start_time_HH_MM = start_time
    end_time_HH_MM = end_time


    # time convert from 17:45 (HH:MM) to datetime object: ------------------------------
    # Convert start_time and end_time to Unix time using the date from obs_id
    date_str = obs_id[:8]  # Extract the date part from obs_id
    date_format = "%Y%m%d"  # Define the date format
    start_datetime_str = f"{date_str} {start_time}"
    end_datetime_str = f"{date_str} {end_time}"
    datetime_format = "%Y%m%d %H:%M"
    start_datetime = datetime.strptime(start_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    end_datetime = datetime.strptime(end_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    start_unix_time = int(start_datetime.timestamp())
    end_unix_time = int(end_datetime.timestamp())
    start_time = start_unix_time
    end_time = end_unix_time

    # ------------------------------

    obs_wavelength=np.linspace(2794,2806,960)
    
    # getting the data: ------------------------------

    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'PF')
    
    # reducing the time: ------------------------------

    start_time = datetime.fromtimestamp(start_time, tz=timezone.utc)
    end_time = datetime.fromtimestamp(end_time, tz=timezone.utc)

    obs_cls.time_clipping(start_time, end_time)

    # test:
    if (lower_box_limit < 0 or lower_box_limit > 370) or (upper_box_limit < 0 or upper_box_limit > 370) or (lower_box_limit > upper_box_limit):
        print('The box limits are not valid. They should be between 0 and 370 and the lower limit should be smaller than the upper limit.')
        return None

    Data = obs_cls.im_arr_global[:,lower_box_limit:upper_box_limit,:].reshape(-1,960)
    
    Data = utils_kmeans.filter_out_zero_spectra(Data)

    labels = assign_mg2k_centroids(X = Data, centroids = centroids)

    

    # making the plot: ------------------------------
    groups_present_in_this_obs = list(set(labels))

    merged_groups = []
    for group in groups_present_in_this_obs:
        for key, value in merge_dict.items():
            if group in value:
                merged_groups.append(key)
                break
    # Remove duplicates
    merged_groups = list(set(merged_groups))
    print('actual merged groups present in this obs:', merged_groups)
    # Convert merges_groups to integers
    merged_groups = [int(group) for group in merged_groups]

    # Sort merged_groups so that grey medoids appear last
    def is_grey(group, clr_dic):
        # Determine if the color of the medoid is grey (or default grey value)
        return clr_dic.get(str(group), 'grey') == '#bbbbbb' or clr_dic.get(str(group), 'grey') == 'grey'

    # Sort the merged groups based on color, keeping non-grey first
    merged_groups_sorted = sorted(merged_groups, key=lambda group: is_grey(group, clr_dic))
    merged_groups = merged_groups_sorted

    # creating the figure and subplots: ------------------------------

    num_groups = len(merged_groups) + 5 # Add 15 to account for the "All" group which leaves the first row empty
    cols = min(5, num_groups)  # Limit the number of columns to 5 or the number of groups
    rows = (num_groups + cols - 1) // cols  # Calculate the number of rows needed
    subplot_width_inches = 5
    subplot_height_inches = 2
    total_width_inches = cols * subplot_width_inches
    total_height_inches = rows * subplot_height_inches
    fig, axs = plt.subplots(rows, cols, figsize=(total_width_inches, total_height_inches),
                            gridspec_kw={'width_ratios': [subplot_width_inches]*cols,
                                        'height_ratios': [subplot_height_inches]*rows})
    ax = axs.ravel()


    print('groups_present_in_this_obs:', groups_present_in_this_obs)

    # plotting: ------------------------------

    #plot all spectra of the box:
    current_subplot_index = 0 
    ax[current_subplot_index].plot(obs_wavelength, Data.T, color='#bbbbbb', alpha=0.4, zorder=1, label = 'all spectra')  # Transpose the array to align dimensions for plotting
    ax[current_subplot_index].tick_params(axis='both', which='both', length=0)  # Removes tick lines
    ax[current_subplot_index].tick_params(labelleft=False, labelbottom=False)  # Removes tick labels
    ax[current_subplot_index].text(0.01, 0.85, f'All', transform=ax[current_subplot_index].transAxes, size=12)
    ax[current_subplot_index].text(0.9, 0.85, f'#spec:\n{len(Data)}', transform=ax[current_subplot_index].transAxes, size=12, ha='center',  # Center align the text
                                    va='center')
    ax[current_subplot_index].set_ylim(-0.1, 1.3)
    # ------------------------------

    current_subplot_index = cols #so then it starts at the next row to plot again
    for i in range(1,cols):
        ax[i].axis('off')
    
    #plot each merged_group individually:
    for merged_group in merged_groups:
        #print('current merged_group:', merged_group)
        if current_subplot_index >= len(ax):
            break

        ax[current_subplot_index].text(0.01, 0.85, f'{merged_group}', transform=ax[current_subplot_index].transAxes, size=12)

        amount_of_spectra_in_merged_group = 0

        for group in merge_dict[str(merged_group)]:
            #print('current group:', group)
            cluster_data_k = Data[labels == group]
            amount_of_spectra_in_merged_group += len(cluster_data_k)

            if skip_spectra > 0:
                cluster_data_k = cluster_data_k[::skip_spectra]  # Select every nth spectrum for plotting

            ax[current_subplot_index].plot(obs_wavelength, cluster_data_k.T, color='#bbbbbb', alpha=0.4, zorder=1)
            
            ax[current_subplot_index].plot(obs_wavelength, mediods[group], color=clr_dic.get(str(group), 'grey'), linewidth=1, zorder=2)

            ax[current_subplot_index].tick_params(axis='both', which='both', length=0)  # Removes tick lines
            ax[current_subplot_index].tick_params(labelleft=False, labelbottom=False)  # Removes tick labels
            ax[current_subplot_index].set_ylim(-0.1, 1.3)
        
        square = patches.Rectangle((2793.6, 0.8), 1, 0.2, facecolor=clr_dic.get(str(merged_group), 'grey'), edgecolor='black')  # (x, y), width, height
        ax[current_subplot_index].add_patch(square)
        ax[current_subplot_index].text(0.9, 0.85, f'#spec:\n{amount_of_spectra_in_merged_group}',
                                       transform=ax[current_subplot_index].transAxes, size=12, ha='center',  # Center align the text
                                        va='center')
        current_subplot_index += 1



    for k in range(current_subplot_index, len(ax)):
        fig.delaxes(ax[k])
    fig.subplots_adjust(hspace=0, wspace=0)
    title_text_size = 15
    fig.text(0.5, 0.95, f'Mg II h&k mediods and spectra: obs_id: {obs_id}, box with limits: y: [{lower_box_limit}, {upper_box_limit}], time: {start_time_HH_MM} - {end_time_HH_MM}', ha='center', va='center', rotation='horizontal',size=title_text_size)
    fig.text(0.5, 0, 'Wavelength [$\AA$]', ha='center', va='center', rotation='horizontal',size=title_text_size)
    fig.text(0.05, 0.5, 'Normalized intensity', ha='center', va='center', rotation='vertical',size=title_text_size)


    # save the figure as a pdf: ------------------------------
    start_time_HH_MM = start_time_HH_MM.replace(':', '_')
    end_time_HH_MM = end_time_HH_MM.replace(':', '_')
    base_name = figure_save_name
    file_name = f'/sml/jannaschk/Progress_meetings/Progress_27_09/{base_name}_{obs_id}_y_{lower_box_limit}_{upper_box_limit}_time_{start_time_HH_MM}_{end_time_HH_MM}.pdf'
    # Append a number if the file exists
    counter = 1
    while os.path.exists(file_name):
        file_name = f'/sml/jannaschk/Progress_meetings/Progress_27_09/{base_name}_{obs_id}_y_{lower_box_limit}_{upper_box_limit}_time_{start_time_HH_MM}_{end_time_HH_MM}_{counter}.pdf'
        counter += 1
    plt.savefig(file_name, format='png')
    print(f"File saved as '{file_name}'.")



    plt.show()

    return None



In [5]:
# subfunctions for the later analysis function:

def load_data(obs_id, do_k_means_flag,
              group_to_subsample, n_subclusters, raster_pos_to_plot):
    '''
    loads the data:
    - adjust path if necessary (for centroids)
    
    returns:

    spectral_data
    spectral_data_reshaped
    centroids_weighed
    centroids_unweighed
    times
    '''

    #load centroids:
    centroids_path = f"{global_constants['global_save_path']}centroids_150_clusters_SOM.npy"
    centroids_weighed = np.load(centroids_path)
    centroids_unweighed = unweigh_the_triplet_again(centroids_weighed, global_constants['global_triple_weight_factor'])

    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'AR')
                                            # typ = 'PF')
    spectral_data = obs_cls.im_arr_global[:,:,:]
    spectral_data_reshaped = spectral_data.reshape(-1,960) # of the form (t*y, lambda)
    times = obs_cls.times_global[1]

    # transform:
    times_transformed, spectral_data_unweighed_transformed, normalisation_values_transformed = utils_data_prep.transform_arrays(obs_cls.times_global,
                                                                                                                                obs_cls.im_arr_global,
                                                                                                                                obs_cls.norm_vals_global,
                                                                           num_of_raster_pos=obs_cls.num_of_raster_pos, forward=False)


    times_one_raster_only = times_transformed[raster_pos_to_plot,:]
    spectral_data_for_one_raster_pos_only = spectral_data_unweighed_transformed[raster_pos_to_plot,:,:,:]


    if do_k_means_flag: #do kmeans
        total_spectra_data_unweighed = utils_kmeans.filter_out_zero_spectra(spectral_data_reshaped) #filter
        # find the spectra of only that group:
        total_spectra_data_weighed = weigh_the_triplet(total_spectra_data_unweighed, global_constants['global_triple_weight_factor'])
        labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = centroids_weighed)
        spectral_data_of_that_group_weighed = total_spectra_data_weighed[labels == group_to_subsample]
        # run kmeans on those spectra of that group:
        subsampled_centroids_weighed, _, _, _ = mini_batch_k_means(X = spectral_data_of_that_group_weighed,
                                                n_clusters=n_subclusters,
                                                batch_size=1000,
                                                n_init=10, SOM = False, verbose=0)
        print('kmeans was done')
        # Redo the centroids_list:
        new_weighed_centroids = centroids_weighed.copy()
        list_1 = new_weighed_centroids.tolist()
        list_1.pop(group_to_subsample)
        list_1.extend(subsampled_centroids_weighed)
        new_weighed_centroids = np.array(list_1)
        new_unweighed_centroids = unweigh_the_triplet_again(new_weighed_centroids, global_constants['global_triple_weight_factor'])
    else:
        new_weighed_centroids = centroids_weighed
        new_unweighed_centroids = centroids_unweighed
        print('no kmeans was done.')


    return spectral_data, spectral_data_reshaped, spectral_data_for_one_raster_pos_only, centroids_weighed, centroids_unweighed, times, times_one_raster_only, new_weighed_centroids, new_unweighed_centroids


def slit_evolv_plot_with_specific_colours(plot_all_raster_pos_separately,
                                          plot_all_rasters_in_one_plot,
                                          plot_only_one_raster_pos,
                                          raster_pos_to_plot,
                                          obs_id,
                                          centroids_weighed,
                                          start_HH_MM, end_HH_MM,
                                          clr_dic,
                                          vertical_lines_flare_start_times,
                                          flares_on_slit_bools,
                                          extra_title_text):
    '''
    Makes a slit evolution plot over time.
    Specify your colours for each group, and it will colour it accordingly

    Parameters:


    Returns:

    
    '''
    global_marker_size = 4
    print('slit evolv plot with specific colours fct was called')
    # preliminary stuff: ------------------------------
    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'AR')
                                            # typ = 'PF')
    # time cutting: ------------------------------
    date_str = obs_id[:8]
    start_datetime_str = f"{date_str} {start_HH_MM}"
    end_datetime_str = f"{date_str} {end_HH_MM}"
    datetime_format = "%Y%m%d %H:%M"
    start_datetime = datetime.strptime(start_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    end_datetime = datetime.strptime(end_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    obs_cls.time_clipping(start_datetime, end_datetime)
    times = obs_cls.times_global[1]
    # spectral data: ------------------------------
    spectral_data_unweighed = obs_cls.im_arr_global[:,:,:]
    times_transformed, spectral_data_unweighed_transformed, normalisation_values_transformed = utils_data_prep.transform_arrays(obs_cls.times_global, obs_cls.im_arr_global, obs_cls.norm_vals_global,
                                                                           num_of_raster_pos=obs_cls.num_of_raster_pos, forward=False)
    normalisation_values = obs_cls.norm_vals_global
    #flare times transform to unix:
    vertical_lines_flare_start_times_unix = []
    for flare_time in vertical_lines_flare_start_times:
        dt_object = datetime.strptime(flare_time, '%Y-%m-%d %H:%M:%S.%f')
        flare_start_time_unix = dt_object.replace(tzinfo=timezone.utc).timestamp()
        vertical_lines_flare_start_times_unix.append(flare_start_time_unix)
    # make the colours for on and off slit for these times:
    colours_for_flare_times = ['red' if bool else 'darkgrey' for bool in flares_on_slit_bools]



    if plot_all_rasters_in_one_plot: #plot all rasters in one plot:
        print('All raster positions in one plot:')

        # making the colour matrix: ------------------------------ 
        time_length = len(times)
        y_length = spectral_data_unweighed.shape[1]
        colour_matrix = np.full((time_length, y_length), global_constants['fallback_colour'], dtype=object)
        flare_start_indices = [np.abs(times - flare_start_time_unix).argmin() for flare_start_time_unix in vertical_lines_flare_start_times_unix]
        # labeling the data: ------------------------------
        labels = []
        flare_positions_x = []
        flare_positions_y = []
        for time_step in tqdm((range(time_length - 1)), desc='Labeling'):
            blabla = spectral_data_unweighed[time_step, :, :].reshape(-1, 960)
            spectral_data_i_weighed = weigh_the_triplet( blabla , global_constants['global_triple_weight_factor'])
            current_labels = assign_mg2k_centroids(spectral_data_i_weighed, centroids_weighed)
            current_colours = [clr_dic.get(label, global_constants['fallback_colour']) for label in current_labels]
            colour_matrix[time_step, :] = current_colours
            current_normalisation_values = normalisation_values[time_step, :]
            for slit_i in range(len(current_normalisation_values)):
                if current_normalisation_values[slit_i] > global_constants['flare_intensity_threshold']:
                    flare_positions_x.append(time_step)
                    flare_positions_y.append(slit_i)
            labels.append(current_labels)




        groups_present_in_this_obs = list(set(np.concatenate(labels)))
        fig, ax = plt.subplots(figsize=(12, 9))
        # get the real times: (not timesteps)
        times_in_unix = times
        times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
        # create the colourmap with the right colours:
        unique_colors = np.unique(colour_matrix)
        color_to_num = {color: idx for idx, color in enumerate(unique_colors)}
        numeric_matrix = np.vectorize(color_to_num.get)(colour_matrix)
        numeric_matrix = numeric_matrix.T
        cmap = ListedColormap(unique_colors)
        # imshow:
        ax.imshow(numeric_matrix, cmap=cmap, aspect = 'auto', origin = 'lower')
        ax.scatter(flare_positions_x, flare_positions_y, color='orange', marker='s', s=global_marker_size, alpha=0.05, zorder = 5)
        # plot time on the x-axis:
        num_labels = 13
        step = len(times_to_plot) // num_labels
        x_ticks = np.arange(0, len(times_to_plot), step)
        x_labels = [time.strftime('%H:%M') for time in times_to_plot[::step]]
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels)
        # legends:
        # change the clr_dic, so that all not coloured groups are filled in with the fallback colour:
        for centroid_number in range(len(centroids_weighed)):
            if centroid_number not in clr_dic:
                clr_dic[centroid_number] = global_constants['fallback_colour']
        # Step 1: Dynamically create a list of unique colors from clr_dic
        colours = list(set(clr_dic.values())) #+ [global_constants['fallback_colour']]  # This will create a unique set of colors
        sorted_groups = []
        for unique_colour in colours:
            for key in sorted(groups_present_in_this_obs):#go through all the groups in this obs, but the groups are ordered
                if clr_dic[key] == unique_colour:
                    # you found the current unique colour in the clr_dic, and it is a group that is in this obs.
                    sorted_groups.append(key)
        # Step 4: Create the ordered legend_elements list
        legend_elements = [
            Patch(
                facecolor=clr_dic.get(group, global_constants['fallback_colour']),
                edgecolor='black' if clr_dic.get(group, global_constants['fallback_colour']) == 'white' else clr_dic.get(group, global_constants['fallback_colour']),
                label=f'{group}'
            )
            for group in sorted_groups
        ]
        num_cols = 12  # Set this to how many columns you want
        ax.legend(handles=legend_elements, title='Groups', bbox_to_anchor=(0.5, -0.4), loc='upper center', ncol=num_cols)
        # titles:
        ax.set_xlabel('Time')
        ax.set_ylabel('Slit Position')
        title = f'Slit Evolution {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}, All {obs_cls.num_of_raster_pos} Raster Positions'
        ax.set_title(title)
        # show vertical lines for the flare start times:
        for index in flare_start_indices:
            ax.axvline(x=index, color='red', linestyle='--', linewidth=1, zorder = 5)
        plt.tight_layout()
        # Save the plot as a PDF and a PNG
        start_HH_MM = start_HH_MM.replace(':', '_')
        end_HH_MM = end_HH_MM.replace(':', '_')
        title = f'slit_{obs_id}_{start_HH_MM}_till_{end_HH_MM}_{extra_title_text}_all_{obs_cls.num_of_raster_pos}_rasters'
        save_plot(title, 'pdf')
        #save_plot(title, 'png')
        plt.show()


    if plot_only_one_raster_pos: #plot only the raster #raster_pos_to_plot:
        print(f'Only raster position {raster_pos_to_plot}:')


        test_data_only_raster_0 = spectral_data_unweighed_transformed[raster_pos_to_plot, :, :, :]
        normalisation_values_this_raster_pos = normalisation_values_transformed[raster_pos_to_plot, :, :]

        # modify the times:
        times_0 = times_transformed[raster_pos_to_plot, :]
        flare_start_indices = [np.abs(times_0 - flare_start_time_unix).argmin() for flare_start_time_unix in vertical_lines_flare_start_times_unix]
        # making the colour matrix: ------------------------------ 
        time_length = len(times_0) 
        y_length = spectral_data_unweighed.shape[1]
        colour_matrix = np.full((time_length, y_length), global_constants['fallback_colour'], dtype=object)
        # labeling the data: ------------------------------
        labels = []
        flare_positions_x = []
        flare_positions_y = []        
        for time_step in tqdm((range(time_length - 1)), desc='Labeling'):
            blabla = test_data_only_raster_0[time_step, :, :].reshape(-1, 960)
            spectral_data_i_weighed = weigh_the_triplet( blabla , global_constants['global_triple_weight_factor'])
            current_labels = assign_mg2k_centroids(spectral_data_i_weighed, centroids_weighed)
            current_colours = [clr_dic.get(label, global_constants['fallback_colour']) for label in current_labels]
            colour_matrix[time_step, :] = current_colours
            current_normalisation_values = normalisation_values_this_raster_pos[time_step, :]
            for slit_i in range(len(current_normalisation_values)):
                if current_normalisation_values[slit_i] > global_constants['flare_intensity_threshold']:
                    flare_positions_x.append(time_step)
                    flare_positions_y.append(slit_i)
            labels.append(current_labels)

        groups_present_in_this_obs = list(set(np.concatenate(labels)))
        fig, ax = plt.subplots(figsize=(12, 9))
        # get the real times: (not timesteps)
        times_in_unix = [x for x in times_0 if x != 0] # filter out zero times (they appear due to the transformation + time clipping)
        times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
        # create the colourmap with the right colours:
        unique_colors = np.unique(colour_matrix)
        color_to_num = {color: idx for idx, color in enumerate(unique_colors)}
        numeric_matrix = np.vectorize(color_to_num.get)(colour_matrix)
        numeric_matrix = numeric_matrix.T
        cmap = ListedColormap(unique_colors)
        # imshow + scatter for flare location:
        ax.imshow(numeric_matrix, cmap=cmap, aspect = 'auto', origin = 'lower')
        ax.scatter(flare_positions_x, flare_positions_y, color='orange', marker='s', s=global_marker_size, alpha=0.05, zorder = 5)
        # plot time on the x-axis:
        num_labels = 13
        step = len(times_to_plot) // num_labels
        x_ticks = np.arange(0, len(times_to_plot), step)
        x_labels = [time.strftime('%H:%M') for time in times_to_plot[::step]]
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels)
        # legends:
        # change the clr_dic, so that all not coloured groups are filled in with the fallback colour:
        for centroid_number in range(len(centroids_weighed)):
            if centroid_number not in clr_dic:
                clr_dic[centroid_number] = global_constants['fallback_colour']
        # Step 1: Dynamically create a list of unique colors from clr_dic
        colours = list(set(clr_dic.values())) #+ [global_constants['fallback_colour']]  # This will create a unique set of colors
        sorted_groups = []
        for unique_colour in colours:
            for key in sorted(groups_present_in_this_obs):#go through all the groups in this obs, but the groups are ordered
                if clr_dic[key] == unique_colour:
                    # you found the current unique colour in the clr_dic, and it is a group that is in this obs.
                    sorted_groups.append(key)
        # Step 4: Create the ordered legend_elements list
        legend_elements = [
            Patch(
                facecolor=clr_dic.get(group, global_constants['fallback_colour']),
                edgecolor='black' if clr_dic.get(group, global_constants['fallback_colour']) == 'white' else clr_dic.get(group, global_constants['fallback_colour']),
                label=f'{group}'
            )
            for group in sorted_groups
        ]
        num_cols = 12  # Set this to how many columns you want
        ax.legend(handles=legend_elements, title='Groups', bbox_to_anchor=(0.5, -0.4), loc='upper center', ncol=num_cols)
        # titles:
        ax.set_xlabel('Time')
        ax.set_ylabel('Slit Position')
        title = f'Slit Evolution {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}, Raster position: {raster_pos_to_plot} / {obs_cls.num_of_raster_pos - 1}'
        ax.set_title(title)
        # show vertical lines for the flare start times:
        flare_start_time_counter = 0
        for index in flare_start_indices:
            flare_start_colour = colours_for_flare_times[flare_start_time_counter] #red if flare, darkgrey if not
            ax.axvline(x=index, color=flare_start_colour, linestyle='--', linewidth=1, zorder = 10)
            flare_start_time_counter += 1
        plt.tight_layout()
        # Save the plot as a PDF and a PNG
        start_HH_MM = start_HH_MM.replace(':', '_')
        end_HH_MM = end_HH_MM.replace(':', '_')
        title = f'slit_{obs_id}_{start_HH_MM}_till_{end_HH_MM}_{extra_title_text}_raster_{raster_pos_to_plot}'
        save_plot(title, 'pdf')
        save_plot(title, 'png')
        plt.show()

 

    if plot_all_raster_pos_separately: #plot all rasters, but each with its own subplot.


        fig, axs = plt.subplots(obs_cls.num_of_raster_pos, 1, figsize=(12, 3 * obs_cls.num_of_raster_pos))
        
        for i in tqdm(range(obs_cls.num_of_raster_pos), desc='Making subplots for each slit position'):
            
            normalisation_values_this_raster_pos = normalisation_values_transformed[i, :, :]
            test_data_only_raster_i = spectral_data_unweighed_transformed[i,:,:, :]
            # modify the times:
            times_i = times[i::obs_cls.num_of_raster_pos] #also start at i, not at zero.
            flare_start_indices = [np.abs(times_i - flare_start_time_unix).argmin() for flare_start_time_unix in vertical_lines_flare_start_times_unix]
            # making the colour matrix: ------------------------------ 
            time_length = test_data_only_raster_i.shape[0] # if transformed
            y_length = spectral_data_unweighed.shape[1]
            colour_matrix = np.full((time_length, y_length), global_constants['fallback_colour'], dtype=object)
            # labeling the data: ------------------------------
            labels = []
            flare_positions_x = []
            flare_positions_y = []
            for time_step in (range(time_length - 1)):
                blabla = test_data_only_raster_i[time_step, :, :].reshape(-1, 960)
                spectral_data_i_weighed = weigh_the_triplet( blabla , global_constants['global_triple_weight_factor'])
                current_labels = assign_mg2k_centroids(spectral_data_i_weighed, centroids_weighed)
                current_colours = [clr_dic.get(label, global_constants['fallback_colour']) for label in current_labels]
                colour_matrix[time_step, :] = current_colours
                labels.append(current_labels)
                current_normalisation_values = normalisation_values_this_raster_pos[time_step, :]
                for slit_i in range(len(current_normalisation_values)):
                    if current_normalisation_values[slit_i] > global_constants['flare_intensity_threshold']:
                        flare_positions_x.append(time_step)
                        flare_positions_y.append(slit_i)

            groups_present_in_this_obs = list(set(np.concatenate(labels)))
            # get the real times: (not timesteps)
            times_in_unix = times_i
            times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
            # create the colourmap with the right colours:
            unique_colors = np.unique(colour_matrix)
            color_to_num = {color: idx for idx, color in enumerate(unique_colors)}
            numeric_matrix = np.vectorize(color_to_num.get)(colour_matrix)
            numeric_matrix = numeric_matrix.T
            cmap = ListedColormap(unique_colors)
            # imshow:
            axs[i].imshow(numeric_matrix, cmap=cmap, aspect = 'auto', origin = 'lower')
            axs[i].scatter(flare_positions_x, flare_positions_y, color='orange', marker='s', s=global_marker_size, alpha=0.05, zorder = 5)
            # plot time on the x-axis:
            num_labels = 13
            step = len(times_to_plot) // num_labels
            if step == 0:
                step = 1
            x_ticks = np.arange(0, len(times_to_plot), step)
            x_labels = [time.strftime('%H:%M') for time in times_to_plot[::step]]
            axs[i].set_xticks(x_ticks)
            axs[i].set_xticklabels(x_labels)
            # titles:
            axs[i].set_xlabel('Time')
            axs[i].set_ylabel('Slit Position')
            title = f'Raster position: {i}/{obs_cls.num_of_raster_pos - 1}'
            axs[i].set_title(title)
            # show vertical lines for the flare start times:
            flare_start_time_counter = 0
            for index in flare_start_indices:
                flare_start_colour = colours_for_flare_times[flare_start_time_counter] #red if flare, darkgrey if not
                axs[i].axvline(x=index, color=flare_start_colour, linestyle='--', linewidth=1, zorder = 5)
                flare_start_time_counter += 1
            if i == 0:
                title = f'Slit Evolution {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}, Raster positions: 0 to {obs_cls.num_of_raster_pos - 1}, Raster position: {i}/{obs_cls.num_of_raster_pos - 1}'
                axs[i].set_title(title)
        # legends:
        # change the clr_dic, so that all not coloured groups are filled in with the fallback colour:
        for centroid_number in range(len(centroids_weighed)):
            if centroid_number not in clr_dic:
                clr_dic[centroid_number] = global_constants['fallback_colour']
        # Step 1: Dynamically create a list of unique colors from clr_dic
        colours = list(set(clr_dic.values())) #+ [global_constants['fallback_colour']]  # This will create a unique set of colors
        sorted_groups = []
        for unique_colour in colours:
            for key in sorted(groups_present_in_this_obs):#go through all the groups in this obs, but the groups are ordered
                if clr_dic[key] == unique_colour:
                    # you found the current unique colour in the clr_dic, and it is a group that is in this obs.
                    sorted_groups.append(key)
        # Step 4: Create the ordered legend_elements list
        legend_elements = [
            Patch(
                facecolor=clr_dic.get(group, global_constants['fallback_colour']),
                edgecolor='black' if clr_dic.get(group, global_constants['fallback_colour']) == 'white' else clr_dic.get(group, global_constants['fallback_colour']),
                label=f'{group}'
            )
            for group in sorted_groups
        ]
        num_cols = 12  # Set this to how many columns you want
        fig.legend(handles=legend_elements, title='Groups', bbox_to_anchor=(0.5, 0), loc='upper center', ncol=num_cols)
        # show plot:
        plt.tight_layout()
        # Save the plot as a PDF and a PNG
        start_HH_MM = start_HH_MM.replace(':', '_')
        end_HH_MM = end_HH_MM.replace(':', '_')
        title = f'slit_{obs_id}_{start_HH_MM}_till_{end_HH_MM}_{extra_title_text}_all_rasters_0_till_{obs_cls.num_of_raster_pos - 1}'
        save_plot(title, 'pdf')
        #save_plot(title, 'png')
        plt.show()





    return None


def intensity_slit_evolv_plot(obs_id, start_HH_MM, end_HH_MM):
    '''
    it makes a slit evolv plot over time,
    but it shows the maximum intensity of each spectrum in the slit.
    To do this, the normalisation value of jonas data prep code is used.
    '''
    print('Intensity slit evolv plot fct was called.')

    # load data: ------------------------------
    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'PF')
    
    # reducing the time: ------------------------------

    # convert hh_mm times to unix time:
    date_str = obs_id[:8]  # Extract the date part from obs_id
    #date_format = "%Y%m%d"  # Define the date format
    start_datetime_str = f"{date_str} {start_HH_MM}"
    end_datetime_str = f"{date_str} {end_HH_MM}"
    datetime_format = "%Y%m%d %H:%M"
    start_datetime = datetime.strptime(start_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    end_datetime = datetime.strptime(end_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    start_unix_time = int(start_datetime.timestamp())
    end_unix_time = int(end_datetime.timestamp())
    start_time = start_unix_time
    end_time = end_unix_time
    start_time = datetime.fromtimestamp(start_time, tz=timezone.utc)
    end_time = datetime.fromtimestamp(end_time, tz=timezone.utc)
    # time cutting: ------------------------------
    obs_cls.time_clipping(start_time, end_time)

    normalisation_values = obs_cls.norm_vals_global
    times = (obs_cls.times_global)[1]
    times_readable = [datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%H:%M') for ts in times]
    normalisation_values_2d = np.squeeze(normalisation_values) 
    normalisation_values_2d = normalisation_values_2d.T

    fig = plt.figure(figsize=(12, 4))
    plt.imshow(normalisation_values_2d, aspect='auto', cmap='YlOrRd', origin='lower')

    num_ticks = 10  # You can adjust this number depending on how many ticks you want
    tick_positions = np.linspace(0, len(times) - 1, num_ticks).astype(int)  # Evenly spaced tick positions
    tick_labels = np.array(times_readable)[tick_positions]  # Corresponding readable times as labels
    plt.xticks(tick_positions, tick_labels) 
    cbar = plt.colorbar()  # Create the colorbar
    plt.xlabel('Time')
    plt.ylabel('Slit Position')
    plt.title(f'Intensity Plot: obsid: {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}')
    plt.tight_layout()  # Adjust layout so labels don't overlap
    title = f'intensity_plot_{obs_id}_{start_HH_MM}_till_{end_HH_MM}'
    save_plot(title, 'pdf')
    save_plot(title, 'png')
    plt.show()



    return None


def show_spectra_of_the_whole_slit(obs_id,
                                   spectral_data_reshaped, new_weighed_centroids,
                                   new_unweighed_centroids, 
                                   skip_spectra_total_plot,
                                   centroids_weighed, centroids_unweighed,
                                   do_k_means_flag, n_subclusters):

    print('showing spectra of the whole slit fct was called.')
   
    path_to_save = global_constants['global_save_path']


    if do_k_means_flag: #do kmeans
        # spectral data:
        total_spectra_data_unweighed = spectral_data_reshaped
        total_spectra_data_weighed = weigh_the_triplet(spectral_data_reshaped, global_constants['global_triple_weight_factor'])
        #labels:
        new_labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = new_weighed_centroids)
        # mediods:
        closest_mediods = utils_kmeans.find_closest_mediods(centroids = new_weighed_centroids, Data = total_spectra_data_weighed)
        closest_mediods = unweigh_the_triplet_again(closest_mediods, global_constants['global_triple_weight_factor'])

        #now find the list of the groups we don't want to see: (we only want to see the new subgroups)
        list_groups_to_remove = list(range(len(new_unweighed_centroids)))
        # keep only the last new subclusters: the list contains all the group that we don't want to see.
        list_groups_to_remove = list_groups_to_remove[:-n_subclusters]

        file_name_to_save = f'{len(new_weighed_centroids)}_centroids_{obs_id}_whole_slit_with_kmeans'

        print('starting centroids summary:')

        centroid_summary(data = total_spectra_data_unweighed,
                        centroids = new_unweighed_centroids,
                        labels = new_labels,
                        closest_mediods = closest_mediods,
                        show_centroids_too = True,
                        additional_title = f' whole slit: {obs_id}', var_modus = 'above 2', 
                            merge_mode = False,
                            merge_dict = {},
                            list_remove_mode = True, 
                            list_groups_to_remove = list_groups_to_remove,
                            plot_only_histogram = False,
                            plot_group_spectra_as_well = True,
                            plot_no_histogram = True,
                            verbose = 1,
                            skip_spectra = skip_spectra_total_plot,
                            packed_subplots = True,
                            max_cols = 4,
                            file_name_to_save = file_name_to_save,
                            path_to_save = path_to_save)    


    else:
        # spectral data:
        total_spectra_data_unweighed = spectral_data_reshaped
        total_spectra_data_weighed = weigh_the_triplet(spectral_data_reshaped, global_constants['global_triple_weight_factor'])
        #labels:
        new_labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = centroids_weighed)
        # mediods:
        closest_mediods = utils_kmeans.find_closest_mediods(centroids = centroids_weighed, Data = total_spectra_data_weighed)
        closest_mediods = unweigh_the_triplet_again(closest_mediods, global_constants['global_triple_weight_factor'])

        list_groups_to_remove = []

        file_name_to_save = f'{len(centroids_weighed)}_centroids_{obs_id}_whole_slit'

        print('starting centroids summary:')

        centroid_summary(data = total_spectra_data_unweighed,
                        centroids = centroids_unweighed,
                        labels = new_labels,
                        closest_mediods = closest_mediods,
                        show_centroids_too = True,
                        additional_title = f' whole slit: {obs_id}', var_modus = 'above 2', 
                        merge_mode = False,
                        merge_dict = {},
                        list_remove_mode = True, 
                        list_groups_to_remove = list_groups_to_remove,
                        plot_only_histogram = False,
                        plot_group_spectra_as_well = True,
                        plot_no_histogram = True,
                        verbose = 1,
                        skip_spectra = skip_spectra_total_plot,
                        packed_subplots = True,
                        max_cols = 4,
                        file_name_to_save = file_name_to_save,
                        path_to_save = path_to_save)  


    return None


def show_spectra_of_only_subgroups(spectral_data_reshaped, new_weighed_centroids,
                                   new_unweighed_centroids,
                                   n_subclusters, group_to_subsample,
                                   skip_spectra_total_plot):

    print('showing spectra of only the subgroups of the whole slit fct was called.')  

    # spectral data:
    total_spectra_data_unweighed = spectral_data_reshaped
    total_spectra_data_weighed = weigh_the_triplet(spectral_data_reshaped, global_constants['global_triple_weight_factor'])
    #labels:
    new_labels = assign_mg2k_centroids(X = total_spectra_data_weighed, centroids = new_weighed_centroids)
    # mediods:
    closest_mediods = utils_kmeans.find_closest_mediods(centroids = new_weighed_centroids, Data = total_spectra_data_weighed)
    closest_mediods = unweigh_the_triplet_again(closest_mediods, global_constants['global_triple_weight_factor'])

    #now find the list of the groups we don't want to see: (we only want to see the new subgroups)
    list_groups_to_remove = list(range(len(new_unweighed_centroids)))
    # keep only the last new subclusters: the list contains all the group that we don't want to see.
    list_groups_to_remove = list_groups_to_remove[:-n_subclusters]

    print('starting centroids summary:')

    centroid_summary(data = total_spectra_data_unweighed,
                     centroids = new_unweighed_centroids,
                     labels = new_labels,
                     closest_mediods = closest_mediods,
                     show_centroids_too = True,
                     additional_title = f' subsampled group: {group_to_subsample}', var_modus = 'above 2', 
                        merge_mode = False,
                        merge_dict = {},
                        list_remove_mode = True, 
                        list_groups_to_remove = list_groups_to_remove,
                        plot_only_histogram = False,
                        plot_group_spectra_as_well = True,
                        plot_no_histogram = True,
                        verbose = 1,
                        skip_spectra = skip_spectra_total_plot,
                        packed_subplots = True,
                        max_cols = 4,
                        file_name_to_save = f'smth_smth_{obs_id}',
                        path_to_save = global_constants['global_save_path'])    





    return None          


def slit_evolv_plot_with_only_subgroups(obs_id,
                                        plot_all_raster_pos_separately,
                                        plot_all_rasters_in_one_plot,
                                        plot_only_one_raster_pos,
                                        raster_pos_to_plot,
                                        new_centroids_weighed, centroids_weighed,
                                        start_HH_MM, end_HH_MM, clr_dic,
                                        group_to_subsample, n_subclusters,
                                        vertical_lines_flare_start_times):



    print('slit evolv with subgroups was called.')
    path = global_constants['global_save_path'] #directory where to save it

    # preliminary stuff: ------------------------------
    directory = '/sml/jannaschk/IRIS_30_obs_prepared'
    obs_cls = utils_data_prep.load_obs_data(highest_directory = directory,
                                            filename = f'{obs_id}',
                                            line = 'MgIIk',
                                            typ = 'PF')
    # time cutting: ------------------------------
    date_str = obs_id[:8]
    start_datetime_str = f"{date_str} {start_HH_MM}"
    end_datetime_str = f"{date_str} {end_HH_MM}"
    datetime_format = "%Y%m%d %H:%M"
    start_datetime = datetime.strptime(start_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    end_datetime = datetime.strptime(end_datetime_str, datetime_format).replace(tzinfo=timezone.utc)
    obs_cls.time_clipping(start_datetime, end_datetime)
    times = obs_cls.times_global[1]
    # spectral data: ------------------------------
    spectral_data_unweighed = obs_cls.im_arr_global[:,:,:]
    spectral_data_unweighed_transformed = utils_data_prep.transform_arrays(obs_cls.times_global, obs_cls.im_arr_global, obs_cls.norm_vals_global,
                                                                           num_of_raster_pos=obs_cls.num_of_raster_pos, forward=False)[1]


    # colours: ------------------------------
    def make_clr_dic(new_centroids_weighed, centroids_weighed):
        clr_dic = {}

        distinct_colors = [
            '#1f77b4',  # Blue
            '#ff7f0e',  # Orange
            '#2ca02c',  # Green
            '#d62728',  # Red
            '#9467bd',  # Purple
            '#8c564b',  # Brown
            '#e377c2',  # Pink
            '#bcbd22',  # Yellow-green
            '#17becf',  # Cyan
            '#aec7e8',  # Light Blue
            '#ffbb78',  # Light Orange
            '#98df8a',  # Light Green
            '#ff9896',  # Light Red
            '#c5b0d5',  # Light Purple
            '#c49c94',  # Light Brown
            '#f7b6d2',  # Light Pink
            '#dbdb8d',  # Light Yellow-green
            '#9edae5',  # Light Cyan
            '#1f77b4',  # Blue
            '#ff7f0e',  # Orange
            '#2ca02c',  # Green
            '#d62728',  # Red
            '#9467bd',  # Purple
            '#8c564b',  # Brown
            '#e377c2',  # Pink
            '#bcbd22',  # Yellow-green
            '#17becf',  # Cyan
            '#aec7e8',  # Light Blue
            '#ffbb78',  # Light Orange
            '#98df8a',  # Light Green
            '#ff9896',  # Light Red
            '#c5b0d5',  # Light Purple
            '#c49c94',  # Light Brown
            '#f7b6d2',  # Light Pink
            '#dbdb8d',  # Light Yellow-green
            '#9edae5'   # Light Cyan
        ]

        for centroid_number in range(len(new_centroids_weighed)):
            if centroid_number < (len(centroids_weighed)-1):
                clr_dic[centroid_number] = global_constants['fallback_colour']
            else:
                clr_dic[centroid_number] = distinct_colors[centroid_number - len(centroids_weighed)]
        
        return clr_dic
    
    print('clr_dic before:', clr_dic)
    clr_dic = make_clr_dic(new_centroids_weighed, centroids_weighed)
    print('clr_dic after:', clr_dic)


    if plot_all_rasters_in_one_plot:

        print('All raster positions in one plot:')

        # making the colour matrix: ------------------------------ 
        time_length = len(times)
        y_length = spectral_data_unweighed.shape[1]
        colour_matrix = np.full((time_length, y_length), global_constants['fallback_colour'], dtype=object)
        # labeling the data: ------------------------------
        labels = []
        time_step = 0 #we need this because the raster_inds are high numbers.
        for i in tqdm((range(time_length - 1)), desc='Labeling'):
            blabla = spectral_data_unweighed[i, :, :].reshape(-1, 960)
            spectral_data_i_weighed = weigh_the_triplet( blabla , global_constants['global_triple_weight_factor'])
            current_labels = assign_mg2k_centroids(spectral_data_i_weighed, new_centroids_weighed)
            current_colours = [clr_dic.get(label, global_constants['fallback_colour']) for label in current_labels]
            colour_matrix[time_step, :] = current_colours
            labels.append(current_labels)
            time_step += 1
        groups_present_in_this_obs = list(set(np.concatenate(labels)))
        fig, ax = plt.subplots(figsize=(12, 9))
        # get the real times: (not timesteps)
        times_in_unix = times
        times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
        # create the colourmap with the right colours:
        unique_colors = np.unique(colour_matrix)
        color_to_num = {color: idx for idx, color in enumerate(unique_colors)}
        numeric_matrix = np.vectorize(color_to_num.get)(colour_matrix)
        numeric_matrix = numeric_matrix.T
        cmap = ListedColormap(unique_colors)
        # imshow:
        ax.imshow(numeric_matrix, cmap=cmap, aspect = 'auto')
        # plot time on the x-axis:
        num_labels = 13
        step = len(times_to_plot) // num_labels
        x_ticks = np.arange(0, len(times_to_plot), step)
        x_labels = [time.strftime('%H:%M') for time in times_to_plot[::step]]
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels)
        # legends:
        # change the clr_dic, so that all not coloured groups are filled in with the fallback colour:
        for centroid_number in range(len(new_centroids_weighed)):
            if centroid_number not in clr_dic:
                clr_dic[centroid_number] = global_constants['fallback_colour']
        # Step 1: Dynamically create a list of unique colors from clr_dic
        colours = list(set(clr_dic.values())) #+ [global_constants['fallback_colour']]  # This will create a unique set of colors
        sorted_groups = []
        for unique_colour in colours:
            for key in sorted(groups_present_in_this_obs):#go through all the groups in this obs, but the groups are ordered
                if clr_dic[key] == unique_colour:
                    # you found the current unique colour in the clr_dic, and it is a group that is in this obs.
                    sorted_groups.append(key)
        # Step 4: Create the ordered legend_elements list
        legend_elements = [
            Patch(
                facecolor=clr_dic.get(group, global_constants['fallback_colour']),
                edgecolor='black' if clr_dic.get(group, global_constants['fallback_colour']) == 'white' else clr_dic.get(group, global_constants['fallback_colour']),
                label=f'{group}'
            )
            for group in sorted_groups
        ]
        num_cols = 12  # Set this to how many columns you want
        ax.legend(handles=legend_elements, title='Groups', bbox_to_anchor=(0.5, -0.4), loc='upper center', ncol=num_cols)
        # titles:
        ax.set_xlabel('Time')
        ax.set_ylabel('Slit Position')
        title = f'Slit Evolution {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}, All {obs_cls.num_of_raster_pos} Raster Positions'
        ax.set_title(title)
        flare_start_indices = [np.abs(times - flare_start_time.unix).argmin() for flare_start_time in vertical_lines_flare_start_times]
        # show vertical lines for the flare start times:
        for index in flare_start_indices:
            ax.axvline(x=index, color='red', linestyle='--', linewidth=1, zorder = 5)
        # show plot:
        plt.tight_layout()
        plt.show()
        # Save the plot as a PDF and a PNG
        start_HH_MM = start_HH_MM.replace(':', '_')
        end_HH_MM = end_HH_MM.replace(':', '_')
        counter = 1
        base_filename = f'slit_subsampledgroup_{group_to_subsample}_nsubclusters_{n_subclusters}_{obs_id}_{start_HH_MM}_till_{end_HH_MM}_all_{obs_cls.num_of_raster_pos}_rasters'
        pdf_filename = f'{path}{base_filename}.pdf'
        png_filename = f'{path}{base_filename}.png'
        while os.path.exists(pdf_filename) or os.path.exists(png_filename):
            pdf_filename = f'{path}{base_filename}_{counter}.pdf'
            png_filename = f'{path}{base_filename}_{counter}.png'
            counter += 1
        fig.savefig(pdf_filename, format='pdf')
        fig.savefig(png_filename, format='png')


    if plot_only_one_raster_pos: #plot only the raster #raster_pos_to_plot:
        print(f'Only raster position {raster_pos_to_plot}:')

        test_data_only_raster_0 = spectral_data_unweighed_transformed[raster_pos_to_plot,:,:, :]
        # modify the times:
        times_0 = times[raster_pos_to_plot::obs_cls.num_of_raster_pos]
        times = times_0
        # making the colour matrix: ------------------------------ 
        time_length = test_data_only_raster_0.shape[0] # if transformed
        y_length = spectral_data_unweighed.shape[1]
        colour_matrix = np.full((time_length, y_length), global_constants['fallback_colour'], dtype=object)
        # labeling the data: ------------------------------
        labels = []
        time_step = 0 #we need this because the raster_inds are high numbers.
        for i in tqdm((range(time_length - 1)), desc='Labeling'):
            blabla = test_data_only_raster_0[i, :, :].reshape(-1, 960)
            spectral_data_i_weighed = weigh_the_triplet( blabla , global_constants['global_triple_weight_factor'])
            current_labels = assign_mg2k_centroids(spectral_data_i_weighed, new_centroids_weighed)
            current_colours = [clr_dic.get(label, global_constants['fallback_colour']) for label in current_labels]
            colour_matrix[time_step, :] = current_colours
            labels.append(current_labels)
            time_step += 1
        groups_present_in_this_obs = list(set(np.concatenate(labels)))
        fig, ax = plt.subplots(figsize=(12, 9))
        # get the real times: (not timesteps)
        times_in_unix = times
        times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
        # create the colourmap with the right colours:
        unique_colors = np.unique(colour_matrix)
        color_to_num = {color: idx for idx, color in enumerate(unique_colors)}
        numeric_matrix = np.vectorize(color_to_num.get)(colour_matrix)
        numeric_matrix = numeric_matrix.T
        cmap = ListedColormap(unique_colors)
        # imshow:
        ax.imshow(numeric_matrix, cmap=cmap, aspect = 'auto')
        # plot time on the x-axis:
        num_labels = 13
        step = len(times_to_plot) // num_labels
        x_ticks = np.arange(0, len(times_to_plot), step)
        x_labels = [time.strftime('%H:%M') for time in times_to_plot[::step]]
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels)
        # legends:
        # change the clr_dic, so that all not coloured groups are filled in with the fallback colour:
        for centroid_number in range(len(centroids_weighed)):
            if centroid_number not in clr_dic:
                clr_dic[centroid_number] = global_constants['fallback_colour']
        # Step 1: Dynamically create a list of unique colors from clr_dic
        colours = list(set(clr_dic.values())) #+ [global_constants['fallback_colour']]  # This will create a unique set of colors
        sorted_groups = []
        for unique_colour in colours:
            for key in sorted(groups_present_in_this_obs):#go through all the groups in this obs, but the groups are ordered
                if clr_dic[key] == unique_colour:
                    # you found the current unique colour in the clr_dic, and it is a group that is in this obs.
                    sorted_groups.append(key)
        # Step 4: Create the ordered legend_elements list
        legend_elements = [
            Patch(
                facecolor=clr_dic.get(group, global_constants['fallback_colour']),
                edgecolor='black' if clr_dic.get(group, global_constants['fallback_colour']) == 'white' else clr_dic.get(group, global_constants['fallback_colour']),
                label=f'{group}'
            )
            for group in sorted_groups
        ]
        num_cols = 12  # Set this to how many columns you want
        ax.legend(handles=legend_elements, title='Groups', bbox_to_anchor=(0.5, -0.4), loc='upper center', ncol=num_cols)
        # titles:
        ax.set_xlabel('Time')
        ax.set_ylabel('Slit Position')
        title = f'Slit Evolution {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}, Raster position: {raster_pos_to_plot} / {obs_cls.num_of_raster_pos - 1}'
        ax.set_title(title)
        flare_start_indices = [np.abs(times - flare_start_time.unix).argmin() for flare_start_time in vertical_lines_flare_start_times]
        # show vertical lines for the flare start times:
        for index in flare_start_indices:
            ax.axvline(x=index, color='red', linestyle='--', linewidth=1, zorder = 5)
        # show plot:
        plt.tight_layout()
        plt.show()
        # Save the plot as a PDF and a PNG
        start_HH_MM = start_HH_MM.replace(':', '_')
        end_HH_MM = end_HH_MM.replace(':', '_')
        counter = 1
        base_filename = f'slit_subsampledgroup_{group_to_subsample}_nsubclusters_{n_subclusters}_{obs_id}_{start_HH_MM}_till_{end_HH_MM}_raster_{raster_pos_to_plot}'
        pdf_filename = f'{path}{base_filename}.pdf'
        png_filename = f'{path}{base_filename}.png'
        while os.path.exists(pdf_filename) or os.path.exists(png_filename):
            pdf_filename = f'{path}{base_filename}_{counter}.pdf'
            png_filename = f'{path}{base_filename}_{counter}.png'
            counter += 1
        fig.savefig(pdf_filename, format='pdf')
        fig.savefig(png_filename, format='png')


    if plot_all_raster_pos_separately: #plot all rasters, but each with its own subplot.

        print('All raster positions in separate subplots:')
        fig, axs = plt.subplots(obs_cls.num_of_raster_pos, 1, figsize=(12, 3 * obs_cls.num_of_raster_pos))
        
        for i in tqdm(range(obs_cls.num_of_raster_pos), desc='Making subplots for each slit position'):
            
            test_data_only_raster_i = spectral_data_unweighed_transformed[i,:,:, :]
            # modify the times:
            times_i = times[i::obs_cls.num_of_raster_pos] #also start at i, not at zero.
            # making the colour matrix: ------------------------------ 
            time_length = test_data_only_raster_i.shape[0] # if transformed
            y_length = spectral_data_unweighed.shape[1]
            colour_matrix = np.full((time_length, y_length), global_constants['fallback_colour'], dtype=object)
            # labeling the data: ------------------------------
            labels = []
            for time_step in (range(time_length - 1)):
                blabla = test_data_only_raster_i[time_step, :, :].reshape(-1, 960)
                spectral_data_i_weighed = weigh_the_triplet( blabla , global_constants['global_triple_weight_factor'])
                current_labels = assign_mg2k_centroids(spectral_data_i_weighed, new_centroids_weighed)
                current_colours = [clr_dic.get(label, global_constants['fallback_colour']) for label in current_labels]
                colour_matrix[time_step, :] = current_colours
                labels.append(current_labels)
            
            groups_present_in_this_obs = list(set(np.concatenate(labels)))
            # get the real times: (not timesteps)
            times_in_unix = times_i
            times_to_plot = [datetime.fromtimestamp(ts, tz = timezone.utc) for ts in times_in_unix]
            # create the colourmap with the right colours:
            unique_colors = np.unique(colour_matrix)
            color_to_num = {color: idx for idx, color in enumerate(unique_colors)}
            numeric_matrix = np.vectorize(color_to_num.get)(colour_matrix)
            numeric_matrix = numeric_matrix.T
            cmap = ListedColormap(unique_colors)
            # imshow:
            axs[i].imshow(numeric_matrix, cmap=cmap, aspect = 'auto')
            # plot time on the x-axis:
            num_labels = 13
            step = len(times_to_plot) // num_labels
            x_ticks = np.arange(0, len(times_to_plot), step)
            x_labels = [time.strftime('%H:%M') for time in times_to_plot[::step]]
            axs[i].set_xticks(x_ticks)
            axs[i].set_xticklabels(x_labels)
            # titles:
            axs[i].set_xlabel('Time')
            axs[i].set_ylabel('Slit Position')
            title = f'Raster position: {i}/{obs_cls.num_of_raster_pos - 1}'
            axs[i].set_title(title)
            flare_start_indices = [np.abs(times_i - flare_start_time.unix).argmin() for flare_start_time in vertical_lines_flare_start_times]
            # show vertical lines for the flare start times:
            for index in flare_start_indices:
                axs[i].axvline(x=index, color='red', linestyle='--', linewidth=1, zorder = 5)
            if i == 0:
                title = f'Slit Evolution {obs_id}, start: {start_HH_MM}, end: {end_HH_MM}, Raster positions: 0 to {obs_cls.num_of_raster_pos - 1}, Raster position: {i}/{obs_cls.num_of_raster_pos - 1}'
                axs[i].set_title(title)
        # legends:
        # change the clr_dic, so that all not coloured groups are filled in with the fallback colour:
        for centroid_number in range(len(centroids_weighed)):
            if centroid_number not in clr_dic:
                clr_dic[centroid_number] = global_constants['fallback_colour']
        # Step 1: Dynamically create a list of unique colors from clr_dic
        colours = list(set(clr_dic.values())) #+ [global_constants['fallback_colour']]  # This will create a unique set of colors
        sorted_groups = []
        for unique_colour in colours:
            for key in sorted(groups_present_in_this_obs):#go through all the groups in this obs, but the groups are ordered
                if clr_dic[key] == unique_colour:
                    # you found the current unique colour in the clr_dic, and it is a group that is in this obs.
                    sorted_groups.append(key)
        # Step 4: Create the ordered legend_elements list
        legend_elements = [
            Patch(
                facecolor=clr_dic.get(group, global_constants['fallback_colour']),
                edgecolor='black' if clr_dic.get(group, global_constants['fallback_colour']) == 'white' else clr_dic.get(group, global_constants['fallback_colour']),
                label=f'{group}'
            )
            for group in sorted_groups
        ]
        num_cols = 12  # Set this to how many columns you want
        
        fig.legend(handles=legend_elements, title='Groups', bbox_to_anchor=(0.5, 0), loc='upper center', ncol=num_cols)
        # show plot:
        plt.tight_layout()
        plt.show()
        # Save the plot as a PDF and a PNG
        start_HH_MM = start_HH_MM.replace(':', '_')
        end_HH_MM = end_HH_MM.replace(':', '_')
        counter = 1
        base_filename = f'slit_subsampledgroup_{group_to_subsample}_nsubclusters_{n_subclusters}_{obs_id}_{start_HH_MM}_till_{end_HH_MM}_all_rasters_0_till_{obs_cls.num_of_raster_pos - 1}'
        pdf_filename = f'{path}{base_filename}.pdf'
        png_filename = f'{path}{base_filename}.png'
        while os.path.exists(pdf_filename) or os.path.exists(png_filename):
            pdf_filename = f'{path}{base_filename}_{counter}.pdf'
            png_filename = f'{path}{base_filename}_{counter}.png'
            counter += 1
        fig.savefig(pdf_filename, format='pdf', bbox_inches='tight')
        fig.savefig(png_filename, format='png', bbox_inches='tight')



    return None


def do_statistics(times, times_one_raster_only,
                  
                  plot_only_one_raster_pos, plot_all_rasters_in_one_plot,
                  
                  spectral_data, spectral_data_for_one_raster_pos_only,

                  raster_pos_to_plot,

                  obs_id, centroids_weighed, start_middle_end_times):

    print('do statistics fct was called.')

    threshold_value = 0.75

    #start_middle_end_times = ['2015-06-22 17:20:00.000', '2015-06-22 17:50:00.000', '2015-06-22 18:10:00.000']
    print('start_middle_end_times:', start_middle_end_times)

    start_time = start_middle_end_times[0]
    middle_time = start_middle_end_times[1]
    end_time = start_middle_end_times[2]
    start_time_unix = datetime.strptime(start_time, '%Y-%m-%d %H:%M:%S.%f').replace(tzinfo=timezone.utc).timestamp()
    middle_time_unix = datetime.strptime(middle_time, '%Y-%m-%d %H:%M:%S.%f').replace(tzinfo=timezone.utc).timestamp()
    end_time_unix = datetime.strptime(end_time, '%Y-%m-%d %H:%M:%S.%f').replace(tzinfo=timezone.utc).timestamp()

    #change the titles and save names with an extra title




    def do_histograms_etc(spectral_data, times, extra_info_text):


        extra_save_text = extra_info_text.replace(' ', '_')


        start_time_index = np.abs(times - start_time_unix).argmin()
        middle_time_index = np.abs(times - middle_time_unix).argmin()
        end_time_index = np.abs(times - end_time_unix).argmin()

        # --------------------------------------------------------------------------------------------

        start_middle_end_times_together = []
        for time in start_middle_end_times:
            new_time = time.replace('-', '_').replace(':', '_').replace('.', '_').replace(' ', '_T_')
            start_middle_end_times_together.append(new_time)
        time_together_text = f'{obs_id[:-11]}_{start_middle_end_times_together[0][11:-7]}_{start_middle_end_times_together[1][11:-7]}_{start_middle_end_times_together[2][11:-7]}'

        # --------------------------------------------------------------------------------------------

        spectral_data_before = spectral_data[start_time_index:middle_time_index, :, :]
        spectral_data_after = spectral_data[middle_time_index:end_time_index, :, :]
        spectral_data_before_reshaped = spectral_data_before.reshape(-1, 960)
        spectral_data_after_reshaped = spectral_data_after.reshape(-1, 960)
        labels_before = assign_mg2k_centroids(X=weigh_the_triplet(spectral_data_before_reshaped,
                                                                global_constants['global_triple_weight_factor']),
                                            centroids=centroids_weighed)
        labels_after = assign_mg2k_centroids(X=weigh_the_triplet(spectral_data_after_reshaped,
                                                                global_constants['global_triple_weight_factor']),
                                            centroids=centroids_weighed)
        # Count occurrences of each label
        unique_before, counts_before = np.unique(labels_before, return_counts=True)
        label_counts_before = dict(zip(unique_before, counts_before))
        unique_after, counts_after = np.unique(labels_after, return_counts=True)
        label_counts_after = dict(zip(unique_after, counts_after))
        # Ensure all labels from 0 to len(centroids_weighed)-1 are present, with 0 count where missing
        all_labels = np.arange(len(centroids_weighed))  # Generate labels from 0 to len(centroids_weighed) - 1
        label_counts_complete = {
            label: {
                'Count_Before': label_counts_before.get(label, 0),
                'Count_After': label_counts_after.get(label, 0)
            }
            for label in all_labels
        }
        # Create a pandas DataFrame with the counts
        df_combined = pd.DataFrame.from_dict(label_counts_complete, orient='index').reset_index()
        df_combined.columns = ['Group', 'Count_Before', 'Count_After']
        # Calculate time-corrected counts
        df_combined['Total_Count'] = df_combined['Count_Before'] + df_combined['Count_After']
        df_combined['Count_Before_Time_Corrected'] = df_combined['Count_Before'] / spectral_data_before.shape[0]
        df_combined['Count_After_Time_Corrected'] = df_combined['Count_After'] / spectral_data_after.shape[0]
        # Normalize counts using time-corrected values
        df_combined['Total_Time_Corrected'] = df_combined['Count_Before_Time_Corrected'] + df_combined['Count_After_Time_Corrected']
        df_combined['Normalized_Before'] = df_combined['Count_Before_Time_Corrected'] / df_combined['Total_Time_Corrected']
        df_combined['Normalized_After'] = df_combined['Count_After_Time_Corrected'] / df_combined['Total_Time_Corrected']
        # Display the DataFrame
        pd.set_option('display.max_rows', None)  # To display all rows
        pd.set_option('display.max_columns', None)  # To display all columns if necessary
        df_combined_reset = df_combined.reset_index(drop=True)
        #display(df_combined_reset)
        # Sort both 'Normalized_Before' and 'Group' based on 'Normalized_Before'
        df_combined_sorted = df_combined.sort_values(by='Normalized_Before').reset_index(drop=True)
        # Create new columns with sorted values
        df_combined['Sorted_Normalized_Before'] = df_combined_sorted['Normalized_Before']  # Sorted normalized values
        df_combined['Sorted_Group'] = df_combined_sorted['Group']
        df_combined['Sorted_Normalized_After'] = df_combined['Normalized_After'].iloc[df_combined['Sorted_Group'].astype(int)].reset_index(drop=True)




        # plot histogram with absolute group numbers, no comparing: ---------------------------------------------------------------



        fig, axs = plt.subplots(2, 1, figsize=(20, 12))
        axs[0].bar(df_combined['Group'], df_combined['Total_Count'], color='navy', label='Total')
        axs[0].legend()
        axs[0].set_title(f'Total Count: {extra_info_text}')
        axs[1].bar(df_combined['Group'], df_combined['Total_Count'], color='navy', label='Total')
        axs[1].set_yscale('log')
        axs[1].legend()
        axs[1].set_title(f'Total Count (Log Scale): {extra_info_text}')
        plt.tight_layout()
        save_plot(f'total_group_counts_linear_and_log_{time_together_text}_{extra_save_text}', 'pdf')
        save_plot(f'total_group_counts_linear_and_log_{time_together_text}_{extra_save_text}', 'png')
        plt.show()




        # plot histogram with time_corrected group numbers, with comparing: ---------------------------------------------------------------
            


        
        n = len(df_combined)
        bar_height = 0.4
        indices = np.arange(n)
        fig, ax = plt.subplots(figsize=(10, 30))
        ax.barh(indices - bar_height / 2, df_combined['Count_Before_Time_Corrected'], 
                height=bar_height, label='Before', color='darkorchid')
        ax.barh(indices + bar_height / 2, df_combined['Count_After_Time_Corrected'], 
                height=bar_height, label='After', color='lightgrey')
        ax.set_xscale('log')
        ax.set_yticks(indices)
        ax.set_yticklabels(df_combined['Group'])
        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top')
        ax.set_xlabel('Time-Corrected Counts (Log Scale)')
        ax.set_ylabel('Group')
        ax.set_title(f'Time-Corrected Group Counts: Before vs After (Log Scale), {extra_info_text}')
        ax.legend()
        plt.tight_layout()
        save_plot(f'time_corrected_group_counts_before_after_{time_together_text}_{extra_save_text}', 'pdf')
        save_plot(f'time_corrected_group_counts_before_after_{time_together_text}_{extra_save_text}', 'png')
        plt.show()





        # Plot combined histogram: -----------------------------------------------------------------------------------------------




        plt.figure(figsize=(20, 6))
        plt.bar(df_combined['Group'], df_combined['Normalized_Before'], color='darkorchid', label='Before')
        plt.bar(df_combined['Group'], df_combined['Normalized_After'], 
                bottom=df_combined['Normalized_Before'], color='white', label='After')
        plt.xlabel('Group')
        plt.ylabel('Proportion')
        plt.title(f'Histogram of Groups: start: {start_time[11:-7]}, split time: {middle_time[11:-7]}, end: {end_time[11:-7]}, {extra_info_text}')
        # Customize x-ticks:
        group_numbers = df_combined['Group']
        plt.xticks(ticks=group_numbers, labels=['' if i % 5 != 0 else str(i) for i in group_numbers], ha='center')
        plt.axhline(y=threshold_value, color='red', linestyle='-', label=f'Threshold: {int(threshold_value * 100)} %', zorder = 5)
        plt.legend(loc = 'upper left')
        # Save the DataFrame as a CSV file
        base_filename = f"{global_constants['global_save_path']}stats_table_group_counts_before_after_{time_together_text}_{extra_save_text}"
        csv_filename = f"{base_filename}.csv"
        counter = 1
        while os.path.exists(csv_filename):
            csv_filename = f"{base_filename}_{counter}.csv"
            counter += 1
        df_combined.to_csv(csv_filename, index=False)
        base_filename = f"histogram_group_counts_before_after_{time_together_text}_{extra_save_text}"
        save_plot(base_filename, 'pdf')
        save_plot(base_filename, 'png')
        plt.show()




        # plot sorted histogram: ---------------------------------------------------------------




        df_combined['Sorted_Group'] = df_combined['Sorted_Group'].astype(str)
        plt.figure(figsize=(5, 30))
        ax = plt.gca()  # Get the current axes
        ax.barh(df_combined['Sorted_Group'], 
                df_combined['Sorted_Normalized_Before'], 
                color='darkorchid', 
                label='Before')
        ax.barh(df_combined['Sorted_Group'], 
                df_combined['Sorted_Normalized_After'], 
                left=df_combined['Sorted_Normalized_Before'],  # Use left instead of bottom
                color='white', 
                label='After')
        ax.xaxis.set_ticks_position('top')  # Move x ticks to the top
        ax.xaxis.tick_top()  # Move the tick labels to the top
        # Change y-ticks to '{group_number}: Nan' for groups with NaN in Sorted_Normalized_Before
        y_tick_labels = []
        for i, value in enumerate(df_combined['Sorted_Normalized_Before']):
            if pd.isna(value):
                y_tick_labels.append(f"{df_combined['Sorted_Group'][i]}: NaN")
            else:
                y_tick_labels.append(df_combined['Sorted_Group'][i])
        ax.set_yticks(np.arange(len(df_combined['Sorted_Group'])))  # Set y-ticks at the correct positions
        ax.set_yticklabels(y_tick_labels)  # Update the y-tick labels
        plt.ylabel('Group')
        plt.xlabel('Proportion')
        ax.xaxis.label.set_position((0.5, 0.05))  # Positioning the label at the top (y=1.05 can be adjusted)
        plt.title(f'Sorted Histogram of Groups by Normalized Before Counts: start: {start_time[11:-7]}, split time: {middle_time[11:-7]}, end: {end_time[11:-7]}, {extra_info_text}')
        plt.axvline(x=threshold_value, color='red', linestyle='-', label=f'Threshold: {int(threshold_value * 100)} %', zorder=5)
        plt.legend(loc='upper left')
        base_filename_sorted = f"histogram_group_counts_sorted_before_after_{time_together_text}_{extra_save_text}"
        save_plot(base_filename_sorted, 'pdf')
        save_plot(base_filename_sorted, 'png')
        plt.show()



        # display the whole table: ---------------------------------------------------------------



        pd.set_option('display.max_rows', None)  # To display all rows
        pd.set_option('display.max_columns', None)  # To display all columns if necessary
        df_combined_reset = df_combined.reset_index(drop=True)
        #display(df_combined_reset)



        # filter groups that meet threshold: ---------------------------------------------------------------



        # Filter groups where the before_normalised_time_corrected proportion is 0.8 or above
        filtered_groups = df_combined[df_combined['Normalized_Before'] >= threshold_value]
        filtered_table = filtered_groups[['Group', 'Normalized_Before']].reset_index(drop=True)
        display(filtered_table)
        # # Save the filtered table as a CSV file
        base_filename = f"{global_constants['global_save_path']}filtered_stats_table_group_counts_before_after_{time_together_text}_threshold_{threshold_value}_{extra_save_text}"
        filtered_csv_filename = f"{base_filename}.csv"
        counter = 1
        while os.path.exists(filtered_csv_filename):
            filtered_csv_filename = f"{base_filename}_{counter}.csv"
            counter += 1
        filtered_table.to_csv(filtered_csv_filename, index=False)




        # distribution plot: -----------------------------------------------------------------------------------------------



        # Plot distribution of Normalized_Before and Normalized_After
        plt.figure(figsize=(10, 6))
        plt.hist(df_combined['Normalized_Before'], bins=20, color='darkorchid', alpha=0.7, edgecolor='black', label='Normalized Before')
        #plt.hist(df_combined['Normalized_After'], bins=20, color='lightgrey', alpha=0.7, edgecolor='black', label='Normalized After')
        plt.xlabel('Proportion')
        plt.ylabel('Frequency')
        plt.title(f'Distribution of Before: start: {start_time[11:-7]}, split time: {middle_time[11:-7]}, end: {end_time[11:-7]}, {extra_info_text}')
        plt.axvline(x=threshold_value, color='red', linestyle='-', label=f'Threshold: {int(threshold_value * 100)} %')
        plt.legend()
        save_plot(f"distribution_{time_together_text}_{extra_save_text}", 'pdf')
        save_plot(f"distribution_{time_together_text}_{extra_save_text}", 'png')
        plt.show()


    if plot_all_rasters_in_one_plot:
        print('all rasters together')
        extra_info_text = 'All rasters together'
        do_histograms_etc(spectral_data, times, extra_info_text)


    if plot_only_one_raster_pos:
        print(f'only one raster position: {raster_pos_to_plot}')
        extra_info_text = f'Only Raster Position {raster_pos_to_plot}'
        do_histograms_etc(spectral_data_for_one_raster_pos_only, times_one_raster_only, extra_info_text)



    return None




In [6]:
# data on which groups are interesting in the goes histos:


'''

# interesting_groups = {
#     '20141022_081850_3860261381': [61, 65, 68, 78, 103, 124, 128,
#                                    133, 138, 141, 142, 145, 148], #done
#     '20140329_140938_3860258481': [12, 30, 56, 65, 96, 101, 105, 110,
#                                    130, 133, 144, 145], #done
#     '20140906_112339_3820259253': [68, 81, 84, 95, 102, 130], #done
# }
# all_numbers = [num for values in interesting_groups.values() for num in values]
# counts = dict(Counter(all_numbers))
# counts_higher_than_one = {num: count for num, count in counts.items() if count > 1}
# print('counts_higher than one',counts_higher_than_one.keys())

# green_flares_to_be_analyzed = [
#   #  "20140329_140938_3860258481", # the third flare is big and on slit. The second is small, on slit. First huge, but next to slit.
#     "20140801_172059_3800013190", # only one big flare on slit
#   #  "20140906_112339_3820259253", # third flare on slit, all others off slit
#   #  "20141022_081850_3860261381", # third flare on slit (current focus)
#     "20150624_111419_3620106017", # second flare big and on slit
#     "20151017_104917_3620257130", # first flare big and on slit
# ]
# flares_on_slit_bools = [
#     # [0, 1, 1], 
#     # [1],
#     # [0, 0, 1, 0],
#     # [0, 0, 1, 0, 0], 
#     # [0, 1],
#     # [1, 0, 0]
# # ]

# analysed_flares_dict = {
#     '20141022_081850_3860261381': {
#                 'before_and_during': [],
#                 'Appearing_before_a_bit_and_lots_during_after': [0, 1, 3, 4, 141, 142],
#                 'Max_count_before_start': [15, 28, 30, 34, 103, 143],
#                 'no_idea': [21, 51, 60, 65, 68, 70, 135, 138, 145],
#                 'very_interesting': []
#                 },

#     '20140329_140938_3860258481': {
#                 'before_and_during': [],
#                 'Appearing_before_a_bit_and_lots_during_after': [],
#                 'Max_count_before_start': [21, 31, 79, 103, 136, ],
#                 'no_idea': [22, 26, 47, 96, ],
#                 'very_interesting': [15, 18, 50, 52, 56, 62, 65, 66, 69, 82, 144, 145, ]
#                 },

#     '20140906_112339_3820259253': {
#                 'before_and_during': [],
#                 'Appearing_before_a_bit_and_lots_during_after': [],
#                 'Max_count_before_start': [98, 102, 120, 133, 145],
#                 'no_idea': [90, 96, 112, 115, 127, 129,],
#                 'very_interesting': [68, 81, 95, ]
#                 },
# }


# # Combine each category across all flares
# combined_before_and_during = []
# combined_appearing_before_and_during_after = []
# combined_max_count_before_start = []
# combined_no_idea = []
# combined_very_interesting = []

# for flare_data in analysed_flares_dict.values():
#     combined_before_and_during.extend(flare_data['before_and_during'])
#     combined_appearing_before_and_during_after.extend(flare_data['Appearing_before_a_bit_and_lots_during_after'])
#     combined_max_count_before_start.extend(flare_data['Max_count_before_start'])
#     combined_no_idea.extend(flare_data['no_idea'])
#     combined_very_interesting.extend(flare_data['very_interesting'])

# supercombined_list = (
#     combined_before_and_during +
#     combined_appearing_before_and_during_after +
#     combined_max_count_before_start +
#     combined_no_idea +
#     combined_very_interesting
# )

# def get_counts_over_one(combined_list):
#     return {element: count for element, count in Counter(combined_list).items() if count > 1}

# counts_before_and_during = get_counts_over_one(combined_before_and_during)
# counts_appearing_before_and_during_after = get_counts_over_one(combined_appearing_before_and_during_after)
# counts_max_count_before_start = get_counts_over_one(combined_max_count_before_start)
# counts_no_idea = get_counts_over_one(combined_no_idea)
# counts_very_interesting = get_counts_over_one(combined_very_interesting)
# counts_supercombined = get_counts_over_one(supercombined_list)

# print("Counts in 'before_and_during' appearing more than once:", counts_before_and_during)
# print("Counts in 'Appearing_before_a_bit_and_lots_during_after' appearing more than once:", counts_appearing_before_and_during_after)
# print("Counts in 'Max_count_before_start' appearing more than once:", counts_max_count_before_start)
# print("Counts in 'no_idea' appearing more than once:", counts_no_idea)
# print("Counts in 'very_interesting' appearing more than once:", counts_very_interesting)
# print("Counts in 'supercombined_list' appearing more than once:", counts_supercombined)



'''



'\n\n# interesting_groups = {\n#     \'20141022_081850_3860261381\': [61, 65, 68, 78, 103, 124, 128,\n#                                    133, 138, 141, 142, 145, 148], #done\n#     \'20140329_140938_3860258481\': [12, 30, 56, 65, 96, 101, 105, 110,\n#                                    130, 133, 144, 145], #done\n#     \'20140906_112339_3820259253\': [68, 81, 84, 95, 102, 130], #done\n# }\n# all_numbers = [num for values in interesting_groups.values() for num in values]\n# counts = dict(Counter(all_numbers))\n# counts_higher_than_one = {num: count for num, count in counts.items() if count > 1}\n# print(\'counts_higher than one\',counts_higher_than_one.keys())\n\n# green_flares_to_be_analyzed = [\n#   #  "20140329_140938_3860258481", # the third flare is big and on slit. The second is small, on slit. First huge, but next to slit.\n#     "20140801_172059_3800013190", # only one big flare on slit\n#   #  "20140906_112339_3820259253", # third flare on slit, all others off slit\n#   #  "2

In [7]:
# The analysis fct:
# combines different analysis tools.

def analysis_change_name_later(
    obs_id,
    start_HH_MM,
    end_HH_MM,
    group_to_subsample,
    n_subclusters,
    skip_spectra_only_subgroups_plot,
    skip_spectra_total_plot,
    start_middle_end_times,

    do_all_the_plots_flag = False,


    slit_evolv_plot_with_specific_colours_flag=False,
    which_group_to_colour = 0,
    plot_all_raster_pos_separately= False,
    plot_all_rasters_in_one_plot = False,
    plot_only_one_raster_pos = False, raster_pos_to_plot = 0,
    vertical_lines_flare_start_times = [],
    flares_on_slit_bools = [],

    slit_evolv_plot_with_only_subgroups_flag=False,

    intensity_slit_evolv_plot_flag=False,
    show_spectra_of_the_whole_slit_flag=False,
    show_spectra_of_only_subgroups_flag=False,
    do_statistics_flag=False,
    goes_curve_with_histogram_flag=False,
    extra_title_text = ''):


    # change pixel setting: above 1000 is long to load pdfs and long to use... so best between 600 and 1000. 300 is the standard.
    plt.rcParams['savefig.dpi'] = 600
    do_k_means_flag = False


    if slit_evolv_plot_with_only_subgroups_flag or show_spectra_of_only_subgroups_flag:
        do_k_means_flag = True

    if do_all_the_plots_flag:
        slit_evolv_plot_with_specific_colours_flag = True
        slit_evolv_plot_with_only_subgroups_flag = True
        intensity_slit_evolv_plot_flag = True
        show_spectra_of_the_whole_slit_flag = True
        show_spectra_of_only_subgroups_flag = True
        do_k_means_flag = True


    # load the data:
    spectral_data, spectral_data_reshaped, spectral_data_for_one_raster_pos_only, centroids_weighed, centroids_unweighed, times, times_one_raster_only, new_weighed_centroids, new_unweighed_centroids = load_data(obs_id = obs_id,
                                                                                                     do_k_means_flag = do_k_means_flag,
                                                                                                     group_to_subsample = group_to_subsample,
                                                                                                     n_subclusters = n_subclusters,
                                                                                                     raster_pos_to_plot = raster_pos_to_plot)
    extra_text = ''
    if do_k_means_flag:
        extra_text = 'And also did kmeans subclustering.'
    print(f'Done loading data. {extra_text}')

    merge_dict = {}
    which_colours_for_which_merged_groups = {}

    if isinstance(which_group_to_colour, int):
        print("which group to colour was a: int")
        merge_dict = {
            f'{which_group_to_colour}': [which_group_to_colour]
            # 83, 85, 111, 148 are the groups that are 80 percent before flare start.
            # flare: 20150622
            # '83': [83],
            # '85': [85],
            # '111': [111],
            # '148': [148],

            # 26, 130 for flare 20150311_151947:
            # '26': [26],
            # '130': [130],

            #downflows
            # '0': [0, 1, 2, 3, 4, 5, 6, 7, 8, 63, 97], 
            #'0': [0],
            # '1': [1],
            # '2': [2],
            # '3': [3],
            # '4': [4],
            # '5': [5],
            # '6': [6],
            # '7': [7],
            # '8': [8],
            # '63': [63],
            # '97': [97],
            #sunspots / quiet sun:
            # '9': [                                       9, 
            #         10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
            #         20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
            #         30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
            #         40,             44, 45, 46, 47, 48, 49,
            #         50, 51, 52, 53, 54, 55, 56, 57, 58, 59,
            #         60, 61, 62,     64, 65, 66, 67, 68, 69,
            #         70, 71, 72, 73, 74, 75, 76, 77, 78, 79,
            #         80, 81, 82,     84,     86, 87, 88, 89,
            #         90, 91, 92, 93, 94, 95, 96,     98, 99,
            #         100, 101, 102, 103, 104, 105, 106, 107, 108, 109,
            #         110,      112, 113, 114, 115, 116, 117, 118, 119,
            #         120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
            #         130, 131, 132, 133, 134,
            #                                                      149],
            #flare:
            # '137': [137, 138, 139, 141, 142, 145, 147],
            # '137': [137],
            # '138': [138],
            # '139': [139],
            # '141': [141],
            # '142': [142],
            #'145': [145],
            # '147': [147],
            #low intensity flares:        
            #'41': [41, 42, 43, 135, 136, 140, 143, 144, 146],
            # '41': [41], 
            # '42': [42],
            # '43': [43],
            # '135': [135],
            # '136': [136],
            # '140': [140],
            # '143': [143],
            # '144': [144],
            # '146': [146],
        }
        which_colours_for_which_merged_groups = { # dont use grey, that is the fallback colour
            f'{which_group_to_colour}': 'purple'
            # 20150622 flare:
            # '83': 'red',
            # '85': 'blue',
            # '111': 'green',
            # '148': 'purple',

            # 20150311_151947 flare:
            # '26': 'red',
            # '130': 'blue',

            # '0': 'purple',
            # '1': 'blue',
            # '2': 'green',
            #'3': 'red',
            # '4': 'purple',
            # '5': 'brown',
            # '6': 'pink',
            # '7': 'yellow',
            # '8': 'cyan',
            # '63': 'black',
            # '97': 'white',

            #'9': 'black',
            #'41': 'red',

            # '137': 'red',
            # '138': 'blue',
            # '139': 'green',
            # '141': 'purple',
            # '142': 'brown',
            #  '145': 'pink',
            # '147': 'yellow',

            #'137': 'purple',
            #'41': 'pink',
            #'0': 'orange',
        }
    

    elif isinstance(which_group_to_colour, list):
        print("which group to colour was a: list")
        merge_dict[f'{which_group_to_colour[0]}'] = which_group_to_colour
        which_colours_for_which_merged_groups = {
            f'{which_group_to_colour[0]}': 'purple'
        }
    
    

    clr_dic = {}
    for key, value in which_colours_for_which_merged_groups.items():
        # key = 13, value = 'green'
        if key not in merge_dict.keys():
            print(f'Error: key {key} is not in merge_dict.keys().')
            continue
        for each_individual_cluster_of_that_group in merge_dict[key]:
            # merge_dict[key] = [13, 14, 15, 16, 30, 31, 32, 33, 34, 35, 39]
            clr_dic[each_individual_cluster_of_that_group] = value
    #print('clr_dic:', clr_dic)


    if slit_evolv_plot_with_specific_colours_flag:
        slit_evolv_plot_with_specific_colours(plot_all_raster_pos_separately = plot_all_raster_pos_separately,
                                              plot_all_rasters_in_one_plot = plot_all_rasters_in_one_plot,
                                              plot_only_one_raster_pos = plot_only_one_raster_pos,
                                              raster_pos_to_plot = raster_pos_to_plot,
                                              obs_id = obs_id,
                                              start_HH_MM = start_HH_MM,
                                              end_HH_MM = end_HH_MM,
                                              centroids_weighed = centroids_weighed,
                                              clr_dic = clr_dic,
                                              vertical_lines_flare_start_times = vertical_lines_flare_start_times,
                                              flares_on_slit_bools = flares_on_slit_bools,
                                              extra_title_text = extra_title_text)

    if slit_evolv_plot_with_only_subgroups_flag:
        slit_evolv_plot_with_only_subgroups(obs_id = obs_id,
                                            plot_all_raster_pos_separately = plot_all_raster_pos_separately,
                                            plot_all_rasters_in_one_plot = plot_all_rasters_in_one_plot,
                                            plot_only_one_raster_pos = plot_only_one_raster_pos,
                                            raster_pos_to_plot = raster_pos_to_plot,
                                            new_centroids_weighed = new_weighed_centroids,
                                            centroids_weighed = centroids_weighed,
                                            start_HH_MM = start_HH_MM, end_HH_MM = end_HH_MM,
                                            clr_dic = clr_dic,
                                            group_to_subsample = group_to_subsample,
                                            n_subclusters = n_subclusters,
                                            vertical_lines_flare_start_times = vertical_lines_flare_start_times)

    if intensity_slit_evolv_plot_flag:
        intensity_slit_evolv_plot(obs_id = obs_id,
                                  start_HH_MM = start_HH_MM,
                                  end_HH_MM = end_HH_MM)

    if show_spectra_of_the_whole_slit_flag:
        show_spectra_of_the_whole_slit(obs_id = obs_id,
                                   spectral_data_reshaped = spectral_data_reshaped,
                                   new_weighed_centroids = new_weighed_centroids,
                                   new_unweighed_centroids = new_unweighed_centroids,
                                   skip_spectra_total_plot = skip_spectra_total_plot,
                                   centroids_weighed = centroids_weighed,
                                   centroids_unweighed = centroids_unweighed,
                                   do_k_means_flag = do_k_means_flag,
                                   n_subclusters = n_subclusters) #still need to code this fct

    if show_spectra_of_only_subgroups_flag:
        show_spectra_of_only_subgroups(spectral_data_reshaped = spectral_data_reshaped,
                                       new_weighed_centroids = new_weighed_centroids,
                                   new_unweighed_centroids = new_unweighed_centroids,
                                   n_subclusters = n_subclusters,
                                   group_to_subsample = group_to_subsample,
                                   skip_spectra_total_plot = skip_spectra_only_subgroups_plot)

    if do_statistics_flag:
        do_statistics(times = times,
                        times_one_raster_only = times_one_raster_only,
                        plot_only_one_raster_pos = plot_only_one_raster_pos,
                        plot_all_rasters_in_one_plot = plot_all_rasters_in_one_plot,
                        spectral_data_for_one_raster_pos_only = spectral_data_for_one_raster_pos_only,
                        spectral_data = spectral_data,
                        obs_id = obs_id, 
                        centroids_weighed = centroids_weighed,
                        start_middle_end_times = start_middle_end_times,
                        raster_pos_to_plot = raster_pos_to_plot)
        

    if goes_curve_with_histogram_flag:
        goes_curve_with_histogram(obs_id = obs_id,
                                  spectral_data = spectral_data,
                                  times = times,
                                  centroids_weighed = centroids_weighed)
            
    return None



In [ ]:
# flare_data_dictionary: dont touch except to add new flares, or more features / keys



flare_data_dict = {
    # the start_HH_MM are taken max 2h before the flare_start of the on_slit flare. or earlier if the flare before is too close.
    # for the end, we take 2h after flare_end

    '20141022_081850_3860261381': { #goes histo done
        'amount_of_flares':  5,
        'flares_on_slit_bools': [False, False, True, False, False],
        'start_HH_MM': '12:00',
        'end_HH_MM': '17:00',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [0, 1, 3, 4, 141, 142],
        'Max_count_before_start': [15, 28, 30, 34, 103, 143],
        'no_idea': [21, 51, 60, 65, 68, 70, 135, 138, 145],
        'very_interesting': []
    },

    '20140329_140938_3860258481': { #goes histo done
        'amount_of_flares':  3,
        'flares_on_slit_bools': [False, True, True],
        'start_HH_MM': '16:30',
        'end_HH_MM': '17:50',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [],
        'Max_count_before_start': [21, 31, 79, 103, 136, ],
        'no_idea': [22, 26, 47, 96, ],
        'very_interesting': [15, 18, 50, 52, 56, 62, 65, 66, 69, 82, 144, 145, ]
    },

    '20140906_112339_3820259253': { #goes histo done
        'amount_of_flares': 4 ,
        'flares_on_slit_bools': [False, False, True, False],
        'start_HH_MM': '14:30',
        'end_HH_MM': '19:00',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [],
        'Max_count_before_start': [98, 102, 120, 133, 145],
        'no_idea': [90, 96, 112, 115, 127, 129,],
        'very_interesting': [68, 81, 95, ]
    },

    '20140801_172059_3800013190': { #goes histo done
        'amount_of_flares': 1 ,
        'flares_on_slit_bools': [True],
        'start_HH_MM': '17:21',
        'end_HH_MM': '20:45',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [],
        'Max_count_before_start': [],
        'no_idea': [],
        'very_interesting': []
    },

    '20151017_104917_3620257130': {
        'amount_of_flares': 3 ,
        'flares_on_slit_bools': [True, False, False],
        'start_HH_MM': '11:00',
        'end_HH_MM': '14:30',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [],
        'Max_count_before_start': [],
        'no_idea': [],
        'very_interesting': []
    },

    '20150624_111419_3620106017': {
        'amount_of_flares': 1 ,
        'flares_on_slit_bools': [True],
        'start_HH_MM': '13:00',
        'end_HH_MM': '16:30',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [],
        'Max_count_before_start': [],
        'no_idea': [],
        'very_interesting': []
    },

    '20151104_125026_3660604042': {
        'amount_of_flares': 1 ,
        'flares_on_slit_bools': [True],
        'start_HH_MM': '12:51',
        'end_HH_MM': '14:00',
        'before_and_during': [],
        'Appearing_before_a_bit_and_lots_during_after': [],
        'Max_count_before_start': [],
        'no_idea': [],
        'very_interesting': []
    },

}
# here we add flare start and end times to the flare_dict, as well as creating new GOES data csv files.



for obs_id in tqdm(flare_data_dict.keys()):
    print('current obs_id:', obs_id)
    # --------------------------------------------------
    gc.collect()
    pth = f'/sml/iris/{obs_id[:4]}/{obs_id[4:6]}/{obs_id[6:8]}/{obs_id}'
    obs = observation( pth, keep_null=True )
    obs_start_ir = obs.start_date
    obs_end_ir = obs.end_date
    obs_start_ir = obs_start_ir.replace('T', ' ')
    obs_end_ir = obs_end_ir.replace('T', ' ')
    timerange = a.Time(obs_start_ir, obs_end_ir)
    hek_query = a.hek.FL & (a.hek.FRM.Name == 'SWPC')
    results = Fido.search(timerange, hek_query)
    hek_results = results['hek']
    # print(f'Number of flares: {len(hek_results)}')
    flare_start_times = []
    flare_end_times = []
    for flare_number in range(len(hek_results)):
        flares_hek = hek_results[flare_number]
        flare_start_times.append(f'{flares_hek["event_starttime"]}')
        flare_end_times.append(f'{flares_hek["event_endtime"]}')
    goes_results = Fido.search(timerange, a.Instrument.xrs & a.goes.SatelliteNumber(15))
    downloaded_files = Fido.fetch(goes_results)
    goes_ts = TimeSeries(downloaded_files)
    times_goes = goes_ts.data.index
    xrsa_data = goes_ts.data['xrsa']
    xrsb_data = goes_ts.data['xrsb']
    length = len(xrsa_data)
    flare_start_times_extended = flare_start_times + [0] * (length - len(flare_start_times))
    flare_end_times_extended = flare_end_times + [0] * (length - len(flare_end_times))
    df_goes = pd.DataFrame({
        'time': times_goes,  # times_goes is the datetime index
        'xrsa': xrsa_data,   # xrsa_data is the first set of GOES data (float)
        'xrsb': xrsb_data,    # xrsb_data is the second set of GOES data (float)
        'flare_start_times': flare_start_times_extended,
        'flare_end_times': flare_end_times_extended
    })
    path_to_goes_data = f'/sml/jannaschk/GOES_data/'
    df_goes.to_csv(f'{path_to_goes_data}goes_data_{obs_id}.csv', index=False)
    amount_of_flares = flare_data_dict[obs_id]['amount_of_flares']
    path_to_goes_data = f'/sml/jannaschk/GOES_data/'
    df_goes = pd.read_csv(f'{path_to_goes_data}goes_data_{obs_id}.csv')
    flare_start_times = df_goes['flare_start_times'][:amount_of_flares].to_list()
    flare_end_times = df_goes['flare_end_times'][:amount_of_flares].to_list()
    flare_data_dict[obs_id]['flare_start_times'] = flare_start_times
    flare_data_dict[obs_id]['flare_end_times'] = flare_end_times


'''

# # Combine each category across all flares
# combined_before_and_during = []
# combined_appearing_before_and_during_after = []
# combined_max_count_before_start = []
# combined_no_idea = []
# combined_very_interesting = []

# for flare_data in flare_data_dict.values():
#     combined_before_and_during.extend(flare_data['before_and_during'])
#     combined_appearing_before_and_during_after.extend(flare_data['Appearing_before_a_bit_and_lots_during_after'])
#     combined_max_count_before_start.extend(flare_data['Max_count_before_start'])
#     combined_no_idea.extend(flare_data['no_idea'])
#     combined_very_interesting.extend(flare_data['very_interesting'])

# supercombined_list = (
#     combined_before_and_during +
#     combined_appearing_before_and_during_after +
#     combined_max_count_before_start +
#     combined_no_idea +
#     combined_very_interesting
# )

# def get_counts_over_one(combined_list):
#     return {element: count for element, count in Counter(combined_list).items() if count > 1}

# counts_before_and_during = get_counts_over_one(combined_before_and_during)
# counts_appearing_before_and_during_after = get_counts_over_one(combined_appearing_before_and_during_after)
# counts_max_count_before_start = get_counts_over_one(combined_max_count_before_start)
# counts_no_idea = get_counts_over_one(combined_no_idea)
# counts_very_interesting = get_counts_over_one(combined_very_interesting)
# counts_supercombined = get_counts_over_one(supercombined_list)

# print("Counts in 'before_and_during' appearing more than once:", counts_before_and_during)
# print("Counts in 'Appearing_before_a_bit_and_lots_during_after' appearing more than once:", counts_appearing_before_and_during_after)
# print("Counts in 'Max_count_before_start' appearing more than once:", counts_max_count_before_start)
# print("Counts in 'no_idea' appearing more than once:", counts_no_idea)
# print("Counts in 'very_interesting' appearing more than once:", counts_very_interesting)
# print("Counts in 'supercombined_list' appearing more than once:", counts_supercombined)


'''




In [9]:
# # here do the slit evolvs of multiple groups together, according to the google sheet of the six flares.

# # so its one combined slit evolv plot for each flare. so six.
# # maybe do it with different colours first, and then with only one colour.

# # also these clusters depend on similar looking spectra centroids.!!!!
# # obs_ids_to_use = [0, 1, 2, 4, 5, 6] # full list
# # obs_ids_to_use = [1, 2, 4, 5, 6] #flare 0 already done.
# # obs_ids_to_use = [0] #test

# clusters_to_plot = [ #different combos of groups, same colours
    
#     [20, 21, 22, 23, 24, 25, 26, 27, 28, 29], #triplet, thin, low downflow, double peak
#     [30, 31, 32, 33, 34, 35, 36], #triplet, thin, low downflow, single peak
#     [55, 56, 57, 58, 59, 60, 61], #almost as the stuff below
#     [65, 66, 67, 68, 69], #low triplet, low but very right downflow, super thin, single peak
#     [77, 78, 79, 80, 81], # same again
#     [128, 129, 130, 131, 132, 133, 134], # very wide, triplet, double peak mostly
# ]



# for obs_id_counter in obs_ids_to_use:
#     # load dictionary data for the flares:
#     obs_id = list(flare_data_dict.keys())[obs_id_counter]
#     start_HH_MM = flare_data_dict[obs_id]['start_HH_MM']
#     end_HH_MM = flare_data_dict[obs_id]['end_HH_MM']
#     amount_of_flares = flare_data_dict[obs_id]['amount_of_flares']
#     flare_start_times = flare_data_dict[obs_id]['flare_start_times']
#     flares_on_slit_bools = flare_data_dict[obs_id]['flares_on_slit_bools']


#     for list_ in tqdm(clusters_to_plot, desc = f'{obs_id}'):

#         clear_output(wait=True)
#         print('current cluster:', list_)
#         extra_title_text = "groups_" + "_".join(map(str, list_))

#         if obs_id_counter == 2:
#             do_only_one_raster = True
#             do_all_rasters = False
#         else:
#             do_only_one_raster = False
#             do_all_rasters = True
        
#         analysis_change_name_later(obs_id = obs_id,
#                         start_HH_MM = start_HH_MM, end_HH_MM = end_HH_MM,
#                         group_to_subsample = 145,
#                         n_subclusters = 5,
#                         skip_spectra_only_subgroups_plot = 1,
#                         skip_spectra_total_plot = 5,
#                         do_all_the_plots_flag = False,

#                         slit_evolv_plot_with_specific_colours_flag = True,
#                         which_group_to_colour = list_,
#                         plot_all_raster_pos_separately= do_all_rasters,
#                         plot_all_rasters_in_one_plot = do_only_one_raster,
#                         plot_only_one_raster_pos = False, raster_pos_to_plot = 0,
#                         vertical_lines_flare_start_times = flare_start_times,
#                         flares_on_slit_bools = flares_on_slit_bools,
#                         slit_evolv_plot_with_only_subgroups_flag = False,

#                         intensity_slit_evolv_plot_flag = False,
#                         show_spectra_of_the_whole_slit_flag = False,
#                         show_spectra_of_only_subgroups_flag = False,

#                         do_statistics_flag = False,
#                         start_middle_end_times = [],
#                         goes_curve_with_histogram_flag = False,
#                         extra_title_text = extra_title_text)
#         print(f'last cluster: {list_}')
#         gc.collect() #collect memory garbage







In [ ]:
# # here its mulitple groups in one goes histogram
# # in the next cell, it one group per goes histogram



# group_numbers_to_show = [1, 2, 3, 4, 5, 6, 7, 8, 63, 97]
# start_time = start_datetime
# end_time = end_datetime
# times_goes = df_goes['time']







# # Make sure start_time and end_time are datetime objects
# start_time = start_time.astimezone(pytz.UTC) if isinstance(start_time, datetime) else start_time
# end_time = end_time.astimezone(pytz.UTC) if isinstance(end_time, datetime) else end_time
# # Create the figure
# fig, ax = plt.subplots(figsize=(20, 6))
# times_goes = times_goes.tolist()
# times_goes = mdates.date2num(times_goes)
# ax.plot(times_goes, xrsa_data, color='white')
# ax.plot(times_goes, xrsb_data, color='white')
# ax.set_yscale('log')
# # Set the x-axis to use datetime with proper timezone
# ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=8, maxticks=12))
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=pytz.UTC))
# # Convert the start_time and end_time to datetime if necessary
# start_ = Time(parse_time(start_time).datetime).to_datetime()
# end_ = Time(parse_time(end_time).datetime).to_datetime()
# start_time_num = mdates.date2num(start_)
# end_time_num = mdates.date2num(end_)
# ax.set_xlim(start_time_num, end_time_num)
# # Set y-axis ticks and grid
# yticks = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
# ax.set_yticks(yticks)
# ax.set_ylim(1e-9, 1e-2)
# ax.set_title(f'GOES X-ray Flux + Histogram, obs_id: {obs_id}')
# ax.grid(True, which='major', axis='y')
# ax.grid(False, which='minor', axis='y')
# # Plot flare start and end times
# flare_colours = ['firebrick' if bool else 'darkgrey' for bool in flares_on_slit_bools]
# for i, (start, end) in enumerate(zip(flare_start_times, flare_end_times)):
#     color = flare_colours[i]
#     start_dt = Time(parse_time(start).datetime).to_datetime()
#     end_dt = Time(parse_time(end).datetime).to_datetime()
#     start_num = mdates.date2num(start_dt)
#     end_num = mdates.date2num(end_dt)
#     ax.axvline(start_num, color=color)
#     ax.axvline(end_num, color=color)
#     ax.axvspan(start_num, end_num, alpha=0.2, color=color, zorder=-3)
# ax2 = ax.twinx()
# df = pd.DataFrame(labels)
# times_datetime = [datetime.fromtimestamp(ts, tz=pytz.UTC) for ts in times]
# bar_width = 0.001 * (times_datetime[-1] - times_datetime[0]).total_seconds() / (len(times_datetime) * 60)
# times_datetime_naive = [ts.replace(tzinfo=None) for ts in times_datetime]
# for group in group_numbers_to_show:
#     numbers_per_timestep = (df == group).sum(axis=1)
#     ax2.bar(times_datetime_naive, numbers_per_timestep, width=bar_width, alpha=0.7, label=f'Group {group}', zorder=5)
# flare_classes = ['A', 'B', 'C', 'M', 'X']
# time_difference = datetime.strptime(obs_end_ir, "%Y-%m-%d %H:%M:%S.%f")  - datetime.strptime(obs_start_ir, "%Y-%m-%d %H:%M:%S.%f")
# total_obs_length_in_minutes = round(time_difference.total_seconds() / 60)
# positions = [np.sqrt(yticks[i] * yticks[i + 1]) for i in range(1, 6)]
# offset_time = start_time - timedelta(minutes=(2 / 375 * total_obs_length_in_minutes))
# offset_time_num = mdates.date2num(offset_time)
# for i, flare_class in enumerate(flare_classes):
#     ax.text(offset_time_num, positions[i], flare_class, fontsize=12, ha='center', va='center', color='firebrick')
# ax2.set_ylabel('Counts')
# ax2.legend()
# ax.set_xlabel('Time')
# ax.set_ylabel('GOES X-ray Flux [Wm$^{-2}$]')



# title = f"goes_histograms_groups_{obs_id}_{'_'.join(map(str, group_numbers_to_show))}_all_same_plot"
# save_plot(title, 'png')
# save_plot(title, 'pdf')
# plt.show()




In [ ]:
# # and now here do the multiple histograms on subplots:




# start_time = start_datetime
# end_time = end_datetime
# times_goes = df_goes['time']
# times_goes = times_goes.tolist()
# times_goes = mdates.date2num(times_goes)

# df = pd.DataFrame(labels)

# arrays_groups_to_plot = [list(range(i, i+10)) for i in range(0, 150, 10)]



# for group_numbers_to_show in tqdm(arrays_groups_to_plot):
#     print('current group_numbers_to_show:', group_numbers_to_show)
#     gc.collect()

#     # figure:
#     fig, axs = plt.subplots(nrows=len(group_numbers_to_show), ncols=1, figsize=(8, int(10 / 3 * len(group_numbers_to_show))), sharex=True)
#     # flare classes:
#     flare_classes = ['A', 'B', 'C', 'M', 'X']
#     time_difference = datetime.strptime(obs_end_ir, "%Y-%m-%d %H:%M:%S.%f")  - datetime.strptime(obs_start_ir, "%Y-%m-%d %H:%M:%S.%f")
#     total_obs_length_in_minutes = round(time_difference.total_seconds() / 60)
#     yticks = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
#     positions = [np.sqrt(yticks[i] * yticks[i + 1]) for i in range(1, 6)]
#     offset_time = start_time - timedelta(minutes=(2 / 375 * total_obs_length_in_minutes))
#     offset_time_num = mdates.date2num(offset_time)
#     # time stuff:
#     start_time = start_time.astimezone(pytz.UTC) if isinstance(start_time, datetime) else start_time
#     end_time = end_time.astimezone(pytz.UTC) if isinstance(end_time, datetime) else end_time
#     start_ = Time(parse_time(start_time).datetime).to_datetime()
#     end_ = Time(parse_time(end_time).datetime).to_datetime()
#     start_time_num = mdates.date2num(start_)
#     end_time_num = mdates.date2num(end_)


#     counter = 0
#     for idx, group in enumerate(group_numbers_to_show):
#         print(f'we are currently doing {counter+1} / {len(group_numbers_to_show)}')
#         counter += 1
#         ax = axs[idx]  # Get the corresponding axis for the group
#         ax.plot(times_goes, xrsa_data, color='white')
#         ax.plot(times_goes, xrsb_data, color='white')
#         ax.set_yscale('log')
#         # Set the x-axis to use datetime with proper timezone
#         ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=8, maxticks=12))
#         ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=pytz.UTC))
#         ax.set_xlim(start_time_num, end_time_num)
#         # Set y-axis ticks and grid
#         ax.set_yticks(yticks)
#         ax.set_ylim(1e-9, 1e-2)
#         ax.set_title(f'GOES X-ray Flux + Histogram, obs_id: {obs_id}')
#         ax.grid(True, which='major', axis='y')
#         ax.grid(False, which='minor', axis='y')
#         # Plot flare start and end times
#         flare_colours = ['firebrick' if bool else 'darkgrey' for bool in flares_on_slit_bools]
#         for i, (start, end) in enumerate(zip(flare_start_times, flare_end_times)):
#             color = flare_colours[i]
#             start_dt = Time(parse_time(start).datetime).to_datetime()
#             end_dt = Time(parse_time(end).datetime).to_datetime()
#             start_num = mdates.date2num(start_dt)
#             end_num = mdates.date2num(end_dt)
#             ax.axvline(start_num, color=color)
#             ax.axvline(end_num, color=color)
#             ax.axvspan(start_num, end_num, alpha=0.2, color=color, zorder=-3)
#         ax2 = ax.twinx()
#         df = pd.DataFrame(labels)
#         times_datetime = [datetime.fromtimestamp(ts, tz=pytz.UTC) for ts in times]
#         bar_width = 0.001 * (times_datetime[-1] - times_datetime[0]).total_seconds() / (len(times_datetime) * 60)
#         times_datetime_naive = [ts.replace(tzinfo=None) for ts in times_datetime]
#         numbers_per_timestep = (df == group).sum(axis=1)
#         ax2.bar(times_datetime_naive, numbers_per_timestep, width=bar_width, alpha=0.7, label=f'Group {group}', zorder=5)
#         for i, flare_class in enumerate(flare_classes):
#             ax.text(offset_time_num, positions[i], flare_class, fontsize=12, ha='center', va='center', color='firebrick')
#         ax2.set_ylabel('Counts')
#         ax2.legend()
#         ax.set_ylabel('GOES X-ray Flux [Wm$^{-2}$]')
#     plt.rcParams['savefig.dpi'] = 600  # Set the desired pixels #standard is 300
#     print('now tight layout')
#     plt.tight_layout()
#     groups_as_a_title = '_'.join([str(group) for group in group_numbers_to_show])
#     title = f"goes_histograms_groups_{obs_id}_{groups_as_a_title}_each_group_alone"
#     print('now saving')
#     save_plot(title, 'png')
#     save_plot(title, 'pdf')
#     print('all done except plt.show()')
#     plt.show()




